TODO List:
- Look into flexbox froggy to deal with running off the screen
- Update sentences to be seeded with first 3 words?
- Start looking at how to understand and visualize proof of performance guarantees / bounds

In [ ]:
from IPython.display import display,Image as IpyImage,Markdown
from pathlib import Path

In [ ]:
md_file_name = 'leviathan-2023.md'
Markdown(Path(md_file_name).read_text())

<div class="prose" markdown="1">

<!-- page 0 -->

# Fast Inference from Transformers via Speculative Decoding

**Yaniv Leviathan** \(^{*1}\) **Matan Kalman** \(^{*1}\) **Yossi Matias** \(^{1}\)

### Abstract

Inference from large autoregressive models like Transformers is slow - decoding \(K\) tokens takes \(K\) serial runs of the model. In this work we introduce *speculative decoding* - an algorithm to sample from autoregressive models faster *without any changes to the outputs*, by computing several tokens in parallel. At the heart of our approach lie the observations that (1) hard language-modeling tasks often include easier subtasks that can be approximated well by more efficient models, and (2) using speculative execution and a novel sampling method, we can make exact decoding from the large models faster, by running them in parallel on the outputs of the approximation models, potentially generating several tokens concurrently, and without changing the distribution. Our method can accelerate existing off-the-shelf models without retraining or architecture changes. We demonstrate it on T5-XXL and show a **2X-3X** acceleration compared to the standard T5X implementation, with identical outputs.

## 1. Introduction

Large autoregressive models, notably large Transformers (Vaswani et al., 2017), are much more capable than smaller models, as is evidenced countless times in recent years e.g., in the text or image domains, like GPT-3 (Brown et al., 2020), LaMDA (Thoppilan et al., 2022), Parti (Yu et al., 2022), and PaLM (Chowdhery et al., 2022). Unfortunately, a single decode step from these larger models is significantly slower than a step from their smaller counterparts, and making things worse, these steps are done serially - decoding \(K\) tokens takes \(K\) serial runs of the model.

Given the importance of large autoregressive models and specifically large Transformers, several approaches were developed to make inference from them faster. Some approaches aim to reduce the inference cost for *all* inputs equally (e.g. Hinton et al., 2015; Jaszczur et al., 2021; Hubara et al., 2016; So et al., 2021; Shazeer, 2019). Other approaches stem from the observation that not all inference steps are born alike - some require a very large model, while others can be approximated well by more efficient models. These *adaptive computation* methods (e.g. Han et al., 2021; Sukhbaatar et al., 2019; Schuster et al., 2021; Scardapane et al., 2020; Bapna et al., 2020; Elbayad et al., 2019; Schwartz et al., 2020) aim to use less compute resources for easier inference steps. While many of these solutions have proven extremely effective in practice, they usually require changing the model architecture, changing the training-procedure and re-training the models, and don't maintain identical outputs.

The key observation above, that some inference steps are "harder" and some are "easier", is also a key motivator for our work. We additionally observe that inference from large models is often not bottlenecked on arithmetic operations, but rather on memory bandwidth and communication, so additional computation resources might be available. Therefore we suggest increasing concurrency as a complementary approach to using an adaptive amount of computation. Specifically, we are able to accelerate inference without changing the model architectures, without changing the training-procedures or needing to re-train the models, and without changing the model output distribution. This is accomplished via *speculative execution*.

Speculative execution (Burton, 1985; Hennessy & Patterson, 2012) is an optimization technique, common in processors, where a task is performed in parallel to verifying if it's actually needed - the payoff being increased concurrency. A well-known example of speculative execution is branch prediction. For speculative execution to be effective, we need an efficient mechanism to suggest tasks to execute that are likely to be needed. In this work, we generalize speculative execution to the stochastic setting - where a task *might* be needed with some probability. Applying this to decoding from autoregressive models like Transformers, we sample generations from more efficient *approximation models* as speculative prefixes for the slower *target models*. With a novel sampling method, *speculative sampling*, we maximize the probability of these speculative tasks to

\rule{0.5\textwidth}{0.4pt}  
\(^{*}\)Equal contribution \(^{1}\)Google Research, Mountain View, CA, USA. Correspondence to: Yaniv Leviathan \(<leviathan@google.com>\).

Proceedings of the \(40^{th}\) International Conference on Machine Learning, Honolulu, Hawaii, USA. PMLR 202, 2023. Copyright 2023 by the author(s).

<!-- page 1 -->

![Figure 1](speculative-decoding//1_figure_1.png)

*Figure 1. Our technique illustrated in the case of unconditional language modeling. Each line represents one iteration of the algorithm. The **green** tokens are the suggestions made by the approximation model (here, a GPT-like Transformer decoder with 6M parameters trained on lm1b with 8k tokens) that the target model (here, a GPT-like Transformer decoder with 97M parameters in the same setting) accepted, while the **red** and **blue** tokens are the rejected suggestions and their corrections, respectively. For example, in the first line the target model was run only once, and 5 tokens were generated.*

be accepted, while guaranteeing that the outputs from our system have the same distribution as those from the target model alone. For example, the sentence in Figure 1, consisting of 38 tokens, was generated by our method with only 9 serial runs of a larger target model (97M parameters) thanks to a smaller and more efficient approximation model (6M parameters), while the probability of generating it is unchanged.

We analyze our method in a variety of tasks and model sizes: unconditional generation from a 97M parameter GPT-like model trained on lm1b, English to German translation and news article summarization with an 11B parameters T5-XXL model, and a dialog task with a 137B parameter LaMDA model. We implement our method and compare actual walltimes for T5-XXL to those of the robust T5X implementation (Roberts et al., 2022), showing an out-of-the-box latency improvement of **2X-3X**, without any change to the outputs (Section 4).

Our method is easy to employ in actual production settings, doesn't require training new models, and doesn't change the outputs. Therefore, in common situations where memory bandwidth is the bottleneck, and compute resources are available, it may be a good default to accelerate sampling from autoregressive models like Transformers.

To summarize, our main contributions are: (1) A generalization of speculative execution to the stochastic setting, with a novel sampling method we call *speculative sampling*, and (2) A decoding mechanism we call *speculative decoding* that can accelerate decoding from autoregressive models, without any change to the model architectures, training regimes and output distributions.

## 2. Speculative Decoding

### 2.1. Overview

Let \( M_p \) be the target model, inference from which we're trying to accelerate, and \( p(x_t|x_{<t}) \) the distribution we get from the model for a prefix \( x_{<t} \). Let \( M_q \) be a more efficient approximation model for the same task, and denote by \( q(x_t|x_{<t}) \) the distribution we get from the model for a prefix \( x_{<t} \)<sup>1</sup>. The core idea is to (1) use the more efficient model \( M_q \) to generate \( \gamma \in \mathbb{Z}^+ \) completions (see Section 3.5 for how to optimally choose this parameter), then (2) use the target model \( M_p \) to evaluate all of the guesses and their respective probabilities from \( M_q \) *in parallel*, accepting all those that *can* lead to an identical distribution, and (3) sampling an additional token from an adjusted distribution to fix the first one that was rejected, or to add an additional one if they are all accepted. That way, each parallel run of the target model \( M_p \) will produce at least one new token (so the number of serial runs of the target model can never, even in the worst case, be larger than the simple autoregressive method), but it can potentially generate many new tokens, up to \( \gamma + 1 \), depending on how well \( M_q \) approximates \( M_p \).

### 2.2. Standardized Sampling

First, note that while there are many methods and parameters of sampling, like argmax, top-k, nucleus, and setting a temperature, and popular implementations usually treat them differently at the logits level, they can all easily be cast into standard sampling from an adjusted probability distribution. For example, argmax sampling is equivalent to zeroing out non-max elements of the distribution and normalizing. We can therefore only deal with standard sampling from a

---

<sup>1</sup>We'll use \( p(x) \) to denote \( p(x_t|x_{<t}) \) whenever the prefix \( x_{<t} \) is clear from the context, and similarly for \( q(x) \).

<!-- page 2 -->

## 2.3. Speculative Sampling

To sample \( x \sim p(x) \), we instead sample \( x \sim q(x) \), keeping it if \( q(x) \leq p(x) \), and in case \( q(x) > p(x) \) we reject the sample with probability \( 1 - \frac{p(x)}{q(x)} \) and sample \( x \) again from an adjusted distribution \( p'(x) = norm(max(0, p(x) - q(x))) \) instead. It's easy to show (see Appendix A.1) that for any distributions \( p(x) \) and \( q(x) \), and \( x \) sampled in this way, indeed \( x \sim p(x) \).

Given the distribution \( q(x) \) obtained from running \( M_q \) on a conditioning *prefix*, we can sample a token \( x_1 \sim q(x) \). We then calculate the distribution \( p(x) \) by running \( M_p \) on *prefix* while in parallel speculatively calculating the distribution of the next token \( x_2 \) by running \( M_p \) on \( prefix + [x_1] \). Once both computations complete, we proceed as per above: If \( x_1 \) is rejected, we discard the computation of \( x_2 \) and re-sample \( x_1 \) from an adjusted distribution, and if \( x_1 \) is accepted, we keep both tokens. Algorithm 1 generalizes this idea to sample between 1 and \( \gamma + 1 \) tokens at once.

---
**Algorithm 1** SpeculativeDecodingStep
---
**Inputs:** \( M_p, M_q, prefix \).
▷ Sample \( \gamma \) guesses \( x_{1,...,\gamma} \) from \( M_q \) autoregressively.
**for** \( i = 1 \) **to** \( \gamma \) **do**
\( q_i(x) \leftarrow M_q(prefix + [x_1, \dots, x_{i-1}]) \)
\( x_i \sim q_i(x) \)
**end for**
▷ Run \( M_p \) in parallel.
\( p_1(x), \dots, p_{\gamma+1}(x) \leftarrow M_p(prefix), \dots, M_p(prefix + [x_1, \dots, x_\gamma]) \)
▷ Determine the number of accepted guesses \( n \).
\( r_1 \sim U(0, 1), \dots, r_\gamma \sim U(0, 1) \)
\( n \leftarrow \min(\{i - 1 \mid 1 \leq i \leq \gamma, r_i > \frac{p_i(x)}{q_i(x)}\} \cup \{\gamma\}) \)
▷ Adjust the distribution from \( M_p \) if needed.
\( p'(x) \leftarrow p_{n+1}(x) \)
**if** \( n < \gamma \) **then**
\( p'(x) \leftarrow norm(max(0, p_{n+1}(x) - q_{n+1}(x))) \)
**end if**
▷ Return one token from \( M_p \), and \( n \) tokens from \( M_q \).
\( t \sim p'(x) \)
**return** \( prefix + [x_1, \dots, x_n, t] \)
---

## 3. Analysis

### 3.1. Number of Generated Tokens

Let's analyze the reduction factor in the number of serial calls to the target model, or equivalently, the expected number of tokens produced by a single run of Algorithm 1.

**Definition 3.1.** The *acceptance rate* \( \beta_{x_{<t}} \), given a prefix \( x_{<t} \), is the probability of accepting \( x_t \sim q(x_t|x_{<t}) \) by speculative sampling, as per Section 2.3\(^2\).

\( E(\beta) \) is then a natural measure of how well \( M_q \) approximates \( M_p \). If we make the simplifying assumption that the \( \beta \)s are i.i.d., and denote \( \alpha = E(\beta) \), then the number of tokens produced by a single run of Algorithm 1 is a capped geometric variable, with success probability \( 1 - \alpha \) and cap \( \gamma + 1 \), and the expected number of tokens generated by Algorithm 1 satisfies Equation (1). See Figure 2.

\[ E(\# \text{ generated tokens}) = \frac{1 - \alpha^{\gamma+1}}{1 - \alpha} \tag{1} \]

![Figure 2](speculative-decoding//2_figure_2.png)
*Figure 2. The expected number of tokens generated by Algorithm 1 as a function of \( \alpha \) for various values of \( \gamma \).*

### 3.2. Calculating \( \alpha \)

We'll now derive a simple formula for calculating \( \alpha \) given a prefix and the two models \( M_p \) and \( M_q \). We start by defining a natural divergence \( D_{LK} \):

**Definition 3.2.** \( D_{LK}(p, q) = \sum_x |p(x) - M(x)| = \sum_x |q(x) - M(x)| \) where \( M(x) = \frac{p(x)+q(x)}{2} \).

**Lemma 3.3.** \( D_{LK}(p, q) = 1 - \sum_x \min(p(x), q(x)) \)

**Proof.** \( D_{LK}(p, q) = \sum_x |p(x) - M(x)| = \sum_x \frac{|p-q|}{2} = 1 - \sum_x \frac{p+q-|p-q|}{2} = 1 - \sum_x \min(p(x), q(x)) \) ◻

From Lemma 3.3 we immediately get the following results:

---
\(^2\)As before, we'll omit the \( x_{<t} \) subscript whenever the prefix is clear from the context.

<!-- page 3 -->

### Fast Inference from Transformers via Speculative Decoding

**Corollary 3.4.** \( D_{LK}(p, q) \) *is a symmetric divergence in \( [0, 1] \).*
\( D_{LK}(p, q) = 0 \iff p = q \).
\( D_{LK}(p, q) = 1 \iff p \text{ and } q \text{ have disjoint support} \).

**Theorem 3.5.** \( \beta = 1 - D_{LK}(p, q) \)

*Proof.* \[
\beta = E_{x \sim q(x)} \begin{cases} 1 & q(x) \le p(x) \\ \frac{p(x)}{q(x)} & q(x) > p(x) \end{cases} = E_{x \sim q(x)} \min(1, \frac{p(x)}{q(x)}) = \sum_x \min(p(x), q(x))
\]

Finally we get:
**Corollary 3.6.** \( \alpha = 1 - E(D_{LK}(p, q)) = E(\min(p, q)) \)

See Table 3 for empirically observed \( \alpha \) values in our experiments.

### 3.3. Walltime Improvement

We’ve shown that with the i.i.d. assumption our algorithm reduces the number of calls to the target model by a factor of \( \frac{1-\alpha^{\gamma+1}}{1-\alpha} \). Note that speculative execution in general, and our algorithm in particular, assume that we have enough compute resources to support the increased concurrency (Section 3.4). For the walltime analysis, we’ll assume that we can run \( \gamma + 1 \) concurrent evaluations of \( M_p \) in parallel without increasing the walltime. To get the total walltime improvement, we now consider the cost of running the approximation model \( M_q \).

**Definition 3.7.** Let \( c \), the *cost coefficient*, be the ratio between the time for a single run of \( M_q \) and the time for a single run of \( M_p \).

Note that unlike \( \alpha \) which is an intrinsic property of the models and the task, the value of \( c \) depends on the hardware configuration and software implementation details. In our experiments where \( M_q \) is typically a couple of orders of magnitude smaller than \( M_p \), \( c \) was always less than 0.05 and often negligibly close to 0.

**Theorem 3.8.** *The expected improvement factor in total walltime by Algorithm 1 is \( \frac{1-\alpha^{\gamma+1}}{(1-\alpha)(\gamma c+1)} \).*

*Proof.* Denote the cost of running a single step of \( M_p \) by \( T \). Now, each run of Algorithm 1 costs \( T c \gamma + T \) (for running the approximation model \( M_q \) \( \gamma \) times and running \( M_p \) once) and according to Equation (1) produces \( \frac{1-\alpha^{\gamma+1}}{1-\alpha} \) tokens on average. So the overall expected cost for producing a token with Algorithm 1 is \( \frac{(c \gamma+1)(1-\alpha)}{1-\alpha^{\gamma+1}} T \). Since the cost of producing a single token with the standard decoding algorithm is \( T \), we get the desired result. ◻

Note that Theorem 3.8 assumes long enough generations (for example, since we run \( M_p \) at least once, the improvement factor is capped by the number of generated tokens).

**Corollary 3.9.** *If \( \alpha > c \), there exists \( \gamma \) for which we’ll get an improvement, and the improvement factor will be at least \( \frac{1+\alpha}{1+c} \).*

*Proof.* If we get an improvement for \( \gamma \), we’d also get an improvement for any \( 0 < \gamma^* < \gamma \), so for our method to yield an improvement, we can evaluate Theorem 3.8 for \( \gamma = 1 \), yielding \( \frac{1-\alpha^2}{(1-\alpha)(c+1)} = \frac{1+\alpha}{1+c} \). ◻

### 3.4. Number of Arithmetic Operations

Algorithm 1 does \( \gamma+1 \) runs of \( M_p \) in parallel, so the number of *concurrent* arithmetic operations grows by a factor of \( \gamma+1 \). Now, since Algorithm 1 produces at most \( \gamma+1 \) tokens per run, the *total* number of arithmetic operations might be higher than that of the standard decoding algorithm. When we accept the sample from \( M_q \) the increased concurrency is ‘‘free’’ and the total number of operations isn’t increased⁳. When we reject a guess though, computation is wasted. Let’s now analyze the effect of our method on the total number of arithmetic operations.

**Definition 3.10.** Let \( \hat{c} \) be the ratio of arithmetic operations per token of the approximation model \( M_q \) to that of the target model \( M_p \).

**Theorem 3.11.** *The expected factor of increase in the number of total operations of Algorithm 1 is \( \frac{(1-\alpha)(\gamma\hat{c}+\gamma+1)}{1-\alpha^{\gamma+1}} \).*

*Proof.* Denote by \( \hat{T} \) the number of arithmetic operations done by a standard decoding baseline per token, i.e. the number of operations of a single run of \( M_p \). Then a single iteration of Algorithm 1 costs \( \hat{T}\hat{c}\gamma + \hat{T}(\gamma + 1) \) operations (for \( \gamma \) runs of \( M_q \) and \( \gamma + 1 \) parallel runs of \( M_p \)). Dividing by the expected number of tokens produced by Algorithm 1, i.e. Equation (1), and by \( \hat{T} \), we get the desired result. ◻

If \( \alpha \) is low, the increase in the number of arithmetic operations is high, and vice-versa. Note that for Transformer decoders, the total number of arithmetic operations by Algorithm 1 (not counting runs of \( M_q \)) can be bounded from above by a single run of the same-size Transformer encoder.

Unlike the total number of arithmetic operations, the total number of memory accesses can go down with our method. Specifically, the target model’s weights and KV cache can be read once per execution of Algorithm 1, so the number of memory accesses for reading them shrinks by a factor of \( \frac{1-\alpha^{\gamma+1}}{1-\alpha} \), according to Equation (1).

### 3.5. Choosing \( \gamma \)

Given \( c \) and \( \alpha \) and assuming enough compute resources (see Section 3.4), the optimal \( \gamma \) is the one maximizing the wall-

———————
⁳ Neglecting the cost of \( M_q \).

<!-- page 4 -->

## Fast Inference from Transformers via Speculative Decoding

![Figure 3](speculative-decoding//4_figure_3.png)

*Figure 3. The optimal \( \gamma \) as a function of \( \alpha \) for various values of \( c \).*

time improvement equation (Theorem 3.8): \( \frac{1-\alpha^{\gamma+1}}{(1-\alpha)(\gamma c+1)} \). Since \( \gamma \) is an integer, it can be easily found numerically, see Figure 3.

Table 1 and Figure 4 illustrate the trade-off between inference speed and the total number of arithmetic operations for various values of \( \alpha \) and \( \gamma \), assuming \( c = \hat{c} = 0 \). Figure 5 shows a simplified trace diagram.

*Table 1. The total number of arithmetic operations and the inference speed vs the baseline, for various values of \( \gamma \) and \( \alpha \), assuming \( c = \hat{c} = 0 \).*

| \( \alpha \) | \( \gamma \) | OPERATIONS | SPEED |
| :--- | :--- | :--- | :--- |
| 0.6 | 2 | 1.53X | 1.96X |
| 0.7 | 3 | 1.58X | 2.53X |
| 0.8 | 2 | 1.23X | 2.44X |
| 0.8 | 5 | 1.63X | 3.69X |
| 0.9 | 2 | 1.11X | 2.71X |
| 0.9 | 10 | 1.60X | 6.86X |

Instead of picking a single value for \( \gamma \) based on \( \alpha \), since the \( \beta \)s aren't constant, we could get further improvement by predicting the value of \( \beta \) and accordingly varying the value of \( \gamma \) during the run of Algorithm 1. To get an upper bound on the additional improvement factor, assume we had an oracle for \( \gamma \). We would then have \( E(\# \text{ generated tokens}) = \frac{1}{1-\alpha} \). For typical values of \( c \) and \( \alpha \), and assuming unbounded compute resources, the enhanced walltime improvement factor can be up to \( \sim 60\% \) higher than the improvement factor with a fixed \( \gamma \). We leave exploring this for future work<sup>4</sup>.

### 3.6. Approximation Models

Speculative sampling, and therefore speculative decoding, guarantee an identical output distribution for any choice of approximation model \( M_q \) without restriction (see Appendix A.1). In our experiments, we mostly tested existing off-the-shelf smaller Transformers as the approximation models. Further, we only tested approximation models of the same architecture as the target models \( M_p \) and using the same probability standardization. In this setup, choosing \( M_q \) to be around two orders of magnitude smaller than \( M_p \) usually performed best, balancing \( \alpha \) and \( c \) (Theorem 3.8).

Another type of approximation models, *negligible-cost models*, are those for which \( c \approx 0 \), i.e. approximation models with a negligible cost relative to the target model. In this case, we get an expected walltime improvement of \( \frac{1-\alpha^{\gamma+1}}{1-\alpha} \), which is bounded from above by \( \frac{1}{1-\alpha} \) (we approach equality if \( \gamma \) is large). One interesting type of negligible-cost approximation models are n-gram models, where the evaluation amounts to a table lookup. Interestingly, in empirical tests (Section 4.2) we get non zero \( \alpha \)s even for these trivial n-gram models. For example, for the English-German translation task, with \( M_p \) being T5-XXL 11B and \( M_q \) being a trivial bigram model, we get \( \alpha \approx 0.2 \) which leads to an inference speed improvement factor of 1.25X with \( \gamma = 3 \).

Other simple heuristics can be used as negligible-cost approximation models. For example, in cases where long sequences are likely to repeat, such as for summarization tasks or chat-like interfaces<sup>5</sup>, an approximation model that simply

![Figure 4](speculative-decoding//4_figure_4.png)

*Figure 4. The speedup factor and the increase in number of arithmetic operations as a function of \( \alpha \) for various values of \( \gamma \).*

---
<sup>4</sup>The above bound assumes that we still run \( M_p \) to verify the oracle's predictions. If we skip those verifications the bound doesn't hold and we would get a substantial additional improvement.

<sup>5</sup>E.g. where a user and a language model iterate on content, like text or code ("can you rewrite this story but change the ending", "can you make this function also do X").

<!-- page 5 -->

![Figure 5](speculative-decoding//5_figure_5.png)

*Figure 5. A simplified trace diagram for a full encoder-decoder Transformer stack. The top row shows speculative decoding with \(\gamma = 7\) so each of the calls to \(M_p\) (the purple blocks) is preceded by 7 calls to \(M_q\) (the blue blocks). The yellow block on the left is the call to the encoder for \(M_p\) and the orange block is the call to the encoder for \(M_q\). Likewise the middle row shows speculative decoding with \(\gamma = 3\), and the bottom row shows standard decoding.*

copies tokens from the context in case we find a matching prefix, might yield high values of \(\alpha\). These parameter-less approximation models, have the additional advantage of being even simpler to deploy from a production standpoint.

Another type of approximation models that can be used by speculative decoding are non-autoregressive models, like those from (Stern et al., 2018). Then, instead of the autoregressive loop in Algorithm 1 we'd just call the non-autoregressive model once.

A final example, interesting mostly from a theoretical perspective, is an approximation model which chooses tokens at random, which guarantees some improvement (although very small) for all models \(M_p\).

### 4. Experiments

#### 4.1. Empirical Walltime Improvement

We implement our algorithm and compare it to the implementation in the T5X codebase for accelerating T5-XXL.

**Setup** We test a standard encoder-decoder T5 version 1.1 model (Raffel et al., 2020) on two tasks from the T5 paper: (1) English to German translation fine tuned on WMT EnDe, and (2) Text summarization fine tuned on CCN/DM. For both tasks, we use T5-XXL (11B) for \(M_p\). For the approximation model \(M_q\) we test several existing configurations, namely T5-large (800M), T5-base (250M), and T5-small (77M) (Raffel et al., 2020). We use existing checkpoints for all models. We measure walltime improvements with a batch size of 1 on a single TPU-v4 for both argmax sampling (temp=0) and standard sampling (temp=1).

**Results** Table 2 shows the empirical results from our method. We see that T5-small (77M), with a good balance of \(c\) and \(\alpha\), provides the highest speedup out of the tested approximation models. As expected we see that \(\alpha\) increases with the size of the approximation model. Interestingly, \(\alpha\) and walltime improvement are higher for argmax sampling (temp=0). We observe speedups of 2.6X (temp=1) and 3.4X (temp=0) on the translation task and slightly lower speedups of 2.3X (temp=1) and 3.1X (temp=0) for the summarization task. These empirical results match well with the theoretical predictions, with some variance due to implementation details (see Appendix A.3).

*Table 2. Empirical results for speeding up inference from a T5-XXL 11B model.*

| TASK | \(M_q\) | TEMP | \(\gamma\) | \(\alpha\) | SPEED |
| :--- | :--- | :--- | :--- | :--- | :--- |
| ENDE | T5-SMALL ★ | 0 | 7 | 0.75 | **3.4X** |
| ENDE | T5-BASE | 0 | 7 | 0.8 | 2.8X |
| ENDE | T5-LARGE | 0 | 7 | 0.82 | 1.7X |
| ENDE | T5-SMALL ★ | 1 | 7 | 0.62 | **2.6X** |
| ENDE | T5-BASE | 1 | 5 | 0.68 | 2.4X |
| ENDE | T5-LARGE | 1 | 3 | 0.71 | 1.4X |
| CNNDM | T5-SMALL ★ | 0 | 5 | 0.65 | **3.1X** |
| CNNDM | T5-BASE | 0 | 5 | 0.73 | 3.0X |
| CNNDM | T5-LARGE | 0 | 3 | 0.74 | 2.2X |
| CNNDM | T5-SMALL ★ | 1 | 5 | 0.53 | **2.3X** |
| CNNDM | T5-BASE | 1 | 3 | 0.55 | 2.2X |
| CNNDM | T5-LARGE | 1 | 3 | 0.56 | 1.7X |

#### 4.2. Empirical \(\alpha\) Values

While we only implemented our method for T5, we measured \(\alpha\) values for various tasks, sampling methods, target models \(M_p\), and approximation models \(M_q\). Specifically, we evaluated the expectation from Corollary 3.6 on 10K tokens generated by \(M_p\), for each of the settings below.

**GPT-like (97M params)** We test a decoder-only Transformer model on unconditional language generation, trained on lm1b (Chelba et al., 2013). The model here is a GPT-like Transformer decoder with Gelu activations (Hendrycks & Gimpel, 2016). For \(M_q\) we experimented with a Trans

<!-- page 6 -->

former decoder model with 6M parameters: dim 256, dim feed-forward 1024, 2 layers, 4 attention heads, as well as simple unigram and bigram models. \(M_p\) has 97M parameters: dim 768, dim feed-forward 3072, 12 layers, 12 attention heads. We used Bert tokenization (Devlin et al., 2019) with 8k tokens for all models.

**LaMDA (137B params)** We tested a decoder only LaMDA model on a dialog task (Thoppilan et al., 2022). We used existing checkpoints from LaMDA 137B as \(M_p\) and LaMDA 8B, LaMDA 2B, and LaMDA 100M for \(M_q\).

See Section 4.1 for the setup of the T5-XXL (11B params) model.

Table 3 summarizes the \(\alpha\) values for the tested cases. We observe that approximation models that are a couple of orders of magnitude smaller than the target model tend to produce \(\alpha\) values between 0.5 and 0.9. Interestingly, we also note that for all models, the sharper the adjusted distribution, the higher the \(\alpha\) values. Finally, we note that even trivial unigram and bigram approximations yield non negligible \(\alpha\) values. For example, for the case of English to German translation, the bigram model has an \(\alpha\) value of 0.2, and since \(c = 0\) in this case, yields a 1.25X speed improvement, which is surprisingly high for this trivial approximation model (but is still lower than the speedup we get from using T5-small as the approximation model).

### 5. Related work

The efficiency of inference from large models was studied extensively (Dehghani et al., 2021). Many approaches aim to speed up inference from large models in general, and autoregressive models like Transformers in particular. Numerous techniques try to make inference more efficient for all tokens, e.g. distillation (Hinton et al., 2015), sparcification (Jaszczur et al., 2021), quantization (Hubara et al., 2016), and architecture modification (So et al., 2021; Shazeer, 2019). Closer to our approach are adaptive computation methods which adapt the amount of computation to problem difficulty (Han et al., 2021). Examples include attending to a subset of the inputs (Sukhbaatar et al., 2019), and early exits (Schuster et al., 2021; Scardapane et al., 2020; Bapna et al., 2020; Elbayad et al., 2019; Schwartz et al., 2020). Notably, Wisdom of Committees (Schwartz et al., 2020) leverages off-the-shelf smaller models, but is an adaptive computation approach, and so it uses a heuristic to determine when to stop, losing the guarantee of identical outputs to those of the target models. In general, adaptive computation methods usually learn, either within the model itself or with an auxiliary model, when a computation shortcut can be taken. Usually, these methods save on both inference time and arithmetic operations, but require a change of architecture, a change of training procedure and training custom models or

| \(M_p\) | \(M_q\) | SMPL | \(\alpha\) |
| :--- | :--- | :--- | :--- |
| GPT-LIKE (97M) | UNIGRAM | T=0 | 0.03 |
| GPT-LIKE (97M) | BIGRAM | T=0 | 0.05 |
| GPT-LIKE (97M) | GPT-LIKE (6M) | T=0 | 0.88 |
| GPT-LIKE (97M) | UNIGRAM | T=1 | 0.03 |
| GPT-LIKE (97M) | BIGRAM | T=1 | 0.05 |
| GPT-LIKE (97M) | GPT-LIKE (6M) | T=1 | 0.89 |
| T5-XXL (ENDE) | UNIGRAM | T=0 | 0.08 |
| T5-XXL (ENDE) | BIGRAM | T=0 | 0.20 |
| T5-XXL (ENDE) | T5-SMALL | T=0 | 0.75 |
| T5-XXL (ENDE) | T5-BASE | T=0 | 0.80 |
| T5-XXL (ENDE) | T5-LARGE | T=0 | 0.82 |
| T5-XXL (ENDE) | UNIGRAM | T=1 | 0.07 |
| T5-XXL (ENDE) | BIGRAM | T=1 | 0.19 |
| T5-XXL (ENDE) | T5-SMALL | T=1 | 0.62 |
| T5-XXL (ENDE) | T5-BASE | T=1 | 0.68 |
| T5-XXL (ENDE) | T5-LARGE | T=1 | 0.71 |
| T5-XXL (CNNDM) | UNIGRAM | T=0 | 0.13 |
| T5-XXL (CNNDM) | BIGRAM | T=0 | 0.23 |
| T5-XXL (CNNDM) | T5-SMALL | T=0 | 0.65 |
| T5-XXL (CNNDM) | T5-BASE | T=0 | 0.73 |
| T5-XXL (CNNDM) | T5-LARGE | T=0 | 0.74 |
| T5-XXL (CNNDM) | UNIGRAM | T=1 | 0.08 |
| T5-XXL (CNNDM) | BIGRAM | T=1 | 0.16 |
| T5-XXL (CNNDM) | T5-SMALL | T=1 | 0.53 |
| T5-XXL (CNNDM) | T5-BASE | T=1 | 0.55 |
| T5-XXL (CNNDM) | T5-LARGE | T=1 | 0.56 |
| LAMDA (137B) | LAMDA (100M) | T=0 | 0.61 |
| LAMDA (137B) | LAMDA (2B) | T=0 | 0.71 |
| LAMDA (137B) | LAMDA (8B) | T=0 | 0.75 |
| LAMDA (137B) | LAMDA (100M) | T=1 | 0.57 |
| LAMDA (137B) | LAMDA (2B) | T=1 | 0.71 |
| LAMDA (137B) | LAMDA (8B) | T=1 | 0.74 |

*Table 3. Empirical \(\alpha\) values for various target models \(M_p\), approximation models \(M_q\), and sampling settings. T=0 and T=1 denote argmax and standard sampling respectively.* 6

re-training of existing models. They usually also change the outputs of the model. We note that while many of the methods above improve the memory to arithmetic-operations ratio, in cases where the ratio remains high, these methods and our speculative decoding method might be effective in tandem.

Two prior methods leverage speculative execution for speeding up decoding from autoregressive models. Blockwise Parallel Decoding (Stern et al., 2018) decodes several tokens in parallel, similarly to our work. However, it only supports greedy decoding (temperature=0) and not the general stochastic setting, it requires additional training of a custom model, and focuses on preserving down-stream task quality, instead of guaranteeing identical outputs. Shallow Aggressive Decoding (SAD) (Sun et al., 2021) also decodes several tokens in parallel, similarly to our work. Unlike our work, SAD only supports copying the input to the output.

<!-- page 7 -->

put, and not general approximation models, making it only
suitable for the cases where the inputs and outputs are very
similar like grammatical error correction. In addition, simi-
larly to Blockwise Parallel Decoding, SAD does not support
the general stochastic sampling setting.

After we initially published our work, an independent im-
plementation of speculative decoding (Chen et al., 2023)
showed similar 2X-2.5X improvements on Chinchilla 70B.

### 6. Discussion

We presented *speculative sampling* which enables efficient
*stochastic speculative execution* - i.e. speculative execu-
tion in the stochastic setting. We analyzed its impact on
decoding from autoregressive models like Transformers via
*speculative decoding* and have shown that given enough
compute resources, we get meaningful 2X-3X speedups in
practice vs T5X, a popular optimized implementation.

One limitation of speculative execution in general, and of
speculative decoding in particular, is that latency is im-
proved through increased concurrency at the cost of an in-
creased number of arithmetic operations. Thus, our method
is not helpful for configurations where additional compu-
tation resources are not available. However, in common
cases where additional computation resources are available
(e.g. when memory bandwidth is the bottleneck) our method
provides the speedup with significant benefits: the model
architecture doesn’t change, retraining isn’t required, and
most importantly, *the output distribution is guaranteed to
stay the same*. Our method is easy to implement, and can
be used to speedup inference using out-of-the-box models
without developing and evaluating custom schemes.

There are several directions for follow up research, impor-
tantly, further investigating the compatibility of speculative
decoding with beam search (see Appendix A.4). Also, while
our method yields substantial speedups with existing off-the-
shelf approximation models, greater improvements might
be obtained via custom approximation models (Section 3.6),
such as those with custom architectures (e.g. custom sizes,
non-autoregressive models, or various heuristics) or with
custom training procedures (e.g. standard distillation with
soft targets from \( M_p \), or optimizing \( M_q \) for \( \alpha \) directly). It
could also be interesting to explore a hierarchical version
of the algorithm, where the approximation model is itself
accelerated by an even faster model, which could allow
for more capable approximation models. In this work we
fixed the approximation model and the number of guesses
\( \gamma \) throughout inference, but varying them during inference
could yield additional improvements (Section 3.5). In our

\rule{150pt}{0.5pt}

\(^6\)Note that the outputs from the LaMDA model always go
through a \( Top_{40} \) filter. This has no effect on argmax, but does
have some effect on standard sampling.

experiments we always performed the same standardization
on the distributions generated by the approximation model
as the desired one for the target model (Section 2.2), but fur-
ther improvements might be obtained by applying different
transformations. We tested speculative decoding only in the
text modality, but it might work well in other domains (e.g.
images) which would be interesting to experiment with.

Finally, we note that *stochastic speculative execution* and
*speculative sampling* can be helpful outside the scope of
*speculative decoding* from autoregressive models. For ex-
ample, given two slow functions, \( f(x) \) and \( g(y) \) such that
\( f(x) \) generates a distribution from which \( g \)'s input is sam-
pled, we could use our method to run \( f \) and \( g \) in parallel.
This setup might arise e.g. in physics simulations, or in rein-
forcement learning where \( f \) is a large model that produces a
distribution on actions, and \( g \) is the world simulation, which
would be interesting to explore.

### Acknowledgments

We would like to extend a special thank you to YaGuang Li
for help with everything LaMDA related and for calculating
the LaMDA figures in the paper, and to Blake Hechtman
for great insights and help with XLA. We would also like
to thank the reviewers for insightful comments, as well
as Asaf Aharoni, Reiner Pope, Sasha Goldshtein, Nadav
Sherman, Eyal Segalis, Eyal Molad, Dani Valevski, Daniel
Wasserman, Valerie Nygaard, Danny Vainstein, the LaMDA
and Theta Labs teams at Google, and our families.

### References

Bapna, A., Arivazhagan, N., and Firat, O. Controlling
computation versus quality for neural sequence models.
*ArXiv*, abs/2002.07106, 2020.

Brown, T. B., Mann, B., Ryder, N., Subbiah, M., Kaplan,
J., Dhariwal, P., Neelakantan, A., Shyam, P., Sastry, G.,
Askell, A., Agarwal, S., Herbert-Voss, A., Krueger, G.,
Henighan, T., Child, R., Ramesh, A., Ziegler, D. M., Wu,
J., Winter, C., Hesse, C., Chen, M., Sigler, E., Litwin, M.,
Gray, S., Chess, B., Clark, J., Berner, C., McCandlish,
S., Radford, A., Sutskever, I., and Amodei, D. Lan-
guage models are few-shot learners. In *Proceedings of
the 34th International Conference on Neural Informa-
tion Processing Systems*, NIPS'20, Red Hook, NY, USA,
2020. Curran Associates Inc. ISBN 9781713829546.

Burton, F. W. Speculative computation, parallelism, and
functional programming. *IEEE Transactions on Comput-
ers*, C-34(12):1190–1193, 1985. doi: 10.1109/TC.1985.
6312218.

Chelba, C., Mikolov, T., Schuster, M., Ge, Q., Brants, T.,
Koehn, P. T., and Robinson, T. One billion word bench-

<!-- page 8 -->

mark for measuring progress in statistical language modeling. In *Interspeech*, 2013.

Chen, C., Borgeaud, S., Irving, G., Lespiau, J.-B., Sifre, L., and Jumper, J. M. Accelerating large language model decoding with speculative sampling. *ArXiv, abs/2302.01318*, 2023.

Chowdhery, A., Narang, S., Devlin, J., Bosma, M., Mishra, G., Roberts, A., Barham, P., Chung, H. W., Sutton, C., Gehrmann, S., Schuh, P., Shi, K., Tsvyashchenko, S., Maynez, J., Rao, A., Barnes, P., Tay, Y., Shazeer, N. M., Prabhakaran, V., Reif, E., Du, N., Hutchinson, B. C., Pope, R., Bradbury, J., Austin, J., Isard, M., Gur-Ari, G., Yin, P., Duke, T., Levskaya, A., Ghemawat, S., Dev, S., Michalewski, H., García, X., Misra, V., Robinson, K., Fedus, L., Zhou, D., Ippolito, D., Luan, D., Lim, H., Zoph, B., Spiridonov, A., Sepassi, R., Dohan, D., Agrawal, S., Omernick, M., Dai, A. M., Pillai, T. S., Pellat, M., Lewkowycz, A., Moreira, E., Child, R., Polozov, O., Lee, K., Zhou, Z., Wang, X., Saeta, B., Díaz, M., Firat, O., Catasta, M., Wei, J., Meier-Hellstern, K. S., Eck, D., Dean, J., Petrov, S., and Fiedel, N. Palm: Scaling language modeling with pathways. *ArXiv, abs/2204.02311*, 2022.

Dehghani, M., Arnab, A., Beyer, L., Vaswani, A., and Tay, Y. The efficiency misnomer. *ArXiv, abs/2110.12894*, 2021.

Devlin, J., Chang, M.-W., Lee, K., and Toutanova, K. Bert: Pre-training of deep bidirectional transformers for language understanding. *ArXiv, abs/1810.04805*, 2019.

Elbayad, M., Gu, J., Grave, E., and Auli, M. Depth-adaptive transformer. *ArXiv, abs/1910.10073*, 2019.

Han, Y., Huang, G., Song, S., Yang, L., Wang, H., and Wang, Y. Dynamic neural networks: A survey. *IEEE Transactions on Pattern Analysis and Machine Intelligence*, 44: 7436–7456, 2021.

Hendrycks, D. and Gimpel, K. Bridging nonlinearities and stochastic regularizers with gaussian error linear units. *ArXiv, abs/1606.08415*, 2016.

Hennessy, J. L. and Patterson, D. A. *Computer Architecture: A Quantitative Approach*. Morgan Kaufmann, Amsterdam, 5 edition, 2012. ISBN 978-0-12-383872-8.

Hinton, G. E., Vinyals, O., and Dean, J. Distilling the knowledge in a neural network. *ArXiv, abs/1503.02531*, 2015.

Hubara, I., Courbariaux, M., Soudry, D., El-Yaniv, R., and Bengio, Y. Quantized neural networks: Training neural networks with low precision weights and activations. *ArXiv, abs/1609.07061*, 2016.

Jaszczur, S., Chowdhery, A., Mohiuddin, A., Kaiser, L., Gajewski, W., Michalewski, H., and Kanerva, J. Sparse is enough in scaling transformers. In *Neural Information Processing Systems*, 2021.

Raffel, C., Shazeer, N., Roberts, A., Lee, K., Narang, S., Matena, M., Zhou, Y., Li, W., and Liu, P. J. Exploring the limits of transfer learning with a unified text-to-text transformer. *The Journal of Machine Learning Research*, 21(1):5485–5551, 2020.

Roberts, A., Chung, H. W., Levskaya, A., Mishra, G., Bradbury, J., Andor, D., Narang, S., Lester, B., Gaffney, C., Mohiuddin, A., Hawthorne, C., Lewkowycz, A., Salcianu, A., van Zee, M., Austin, J., Goodman, S., Soares, L. B., Hu, H., Tsvyashchenko, S., Chowdhery, A., Bastings, J., Bulian, J., García, X., Ni, J., Chen, A., Kenealy, K., Clark, J., Lee, S., Garrette, D. H., Lee-Thorp, J., Raffel, C., Shazeer, N. M., Ritter, M., Bosma, M., Passos, A., Maitin-Shepard, J. B., Fiedel, N., Omernick, M., Saeta, B., Sepassi, R., Spiridonov, A., Newlan, J., and Gesmundo, A. Scaling up models and data with t5x and seqio. *ArXiv, abs/2203.17189*, 2022.

Scardapane, S., Scarpiniti, M., Baccarelli, E., and Uncini, A. Why should we add early exits to neural networks? *Cognitive Computation*, 12(5):954–966, 2020.

Schuster, T., Fisch, A., Jaakkola, T., and Barzilay, R. Consistent accelerated inference via confident adaptive transformers. In *Conference on Empirical Methods in Natural Language Processing*, 2021.

Schwartz, R., Stanovsky, G., Swayamdipta, S., Dodge, J., and Smith, N. A. The right tool for the job: Matching model and instance complexities. In *Annual Meeting of the Association for Computational Linguistics*, 2020.

Shazeer, N. M. Fast transformer decoding: One write-head is all you need. *ArXiv, abs/1911.02150*, 2019.

So, D. R., Ma’nke, W., Liu, H., Dai, Z., Shazeer, N. M., and Le, Q. V. Primer: Searching for efficient transformers for language modeling. *ArXiv, abs/2109.08668*, 2021.

Stern, M., Shazeer, N., and Uszkoreit, J. Blockwise parallel decoding for deep autoregressive models. *Advances in Neural Information Processing Systems*, 31, 2018.

Sukhbaatar, S., Grave, E., Bojanowski, P., and Joulin, A. Adaptive attention span in transformers. In *Annual Meeting of the Association for Computational Linguistics*, 2019.

Sun, X., Ge, T., Wei, F., and Wang, H. Instantaneous grammatical error correction with shallow aggressive decoding. *ArXiv, abs/2106.04970*, 2021.

9

<!-- page 9 -->

Thoppilan, R., Freitas, D. D., Hall, J., Shazeer, N. M., Kul-shreshtha, A., Cheng, H.-T., Jin, A., Bos, T., Baker, L., Du, Y., Li, Y., Lee, H., Zheng, H., Ghafouri, A., Menegali, M., Huang, Y., Krikun, M., Lepikhin, D., Qin, J., Chen, D., Xu, Y., Chen, Z., Roberts, A., Bosma, M., Zhou, Y., Chang, C.-C., Krivokon, I. A., Rusch, W. J., Pickett, M., Meier-Hellstern, K. S., Morris, M. R., Doshi, T., Santos, R. D., Duke, T., Søraker, J. H., Zevenbergen, B., Prabhakaran, V., Díaz, M., Hutchinson, B., Olson, K., Molina, A., Hoffman-John, E., Lee, J., Aroyo, L., Rajakumar, R., Butryna, A., Lamm, M., Kuzmina, V. O., Fenton, J., Cohen, A., Bernstein, R., Kurzweil, R., Aguera-Arcas, B., Cui, C., Croak, M., hsin Chi, E. H., and Le, Q. Lamda: Language models for dialog applications. *ArXiv*, abs/2201.08239, 2022.

Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, Ł., and Polosukhin, I. Attention is all you need. *Advances in neural information processing systems*, 30, 2017.

Yu, J., Xu, Y., Koh, J. Y., Luong, T., Baid, G., Wang, Z., Vasudevan, V., Ku, A., Yang, Y., Ayan, B. K., Hutchinson, B. C., Han, W., Parekh, Z., Li, X., Zhang, H., Baldridge, J., and Wu, Y. Scaling autoregressive models for content-rich text-to-image generation. *ArXiv*, abs/2206.10789, 2022.

<!-- page 10 -->

## A. Appendix

### A.1. Correctness of Speculative Sampling

We will now show that for any distributions \( p(x) \) and \( q(x) \), the tokens sampled via *speculative sampling* from \( p(x) \) and \( q(x) \) are distributed identically to those sampled from \( p(x) \) alone. Let \( \beta \) be the acceptance probability (Definition 3.1).

Note that as \( p'(x) = norm(max(0, p(x) - q(x))) = \frac{p(x) - \min(q(x), p(x))}{\sum_{x'} (p(x') - \min(q(x'), p(x')))} = \frac{p(x) - \min(q(x), p(x))}{1 - \beta} \), the normalizing constant for the adjusted distribution \( p'(x) \) is \( 1 - \beta \), where the last equation follows immediately from Lemma 3.3 and Theorem 3.5.

Now:

\[ P(x = x') = P(guess\ accepted, x = x') + P(guess\ rejected, x = x') \]

Where:

\[ P(guess\ accepted, x = x') = q(x') \min(1, \frac{p(x')}{q(x')}) = \min(q(x'), p(x')) \]

And:

\[ P(guess\ rejected, x = x') = (1 - \beta)p'(x') = p(x') - \min(q(x'), p(x')) \]

Overall:

\[ P(x = x') = \min(p(x'), q(x')) + p(x') - \min(p(x'), q(x')) = p(x'). \]

As desired. \(\square\)

### A.2. Speculative Sampling vs. Rejection Sampling

Rejection sampling is the following iterative sampling procedure that looks superficially similar to ours:

1. Sample \( x \sim q(x) \) and \( r \sim U(0, 1) \).
2. If \( r < \frac{p(x)}{M q(x)} \) return \( x \).
3. Go to 1.

Where \( M = \max_x \frac{p(x)}{q(x)} \). We could employ a non-iterative version of rejection sampling instead of speculative sampling - specifically go through steps 1 and 2 above, and otherwise sample from an *unmodified* \( p(x) \) directly. That would be much less efficient than our method though. Specifically, the expected accept probability here is \( E_{x \sim q(x)} \frac{p(x)}{Mq(x)} = \sum_x p(x) \min_{x'} \frac{q(x')}{p(x')} \le \sum_x p(x) \min(1, \frac{q(x)}{p(x)}) = \sum_x \min(p(x), q(x)) = \alpha \) is (potentially much) lower than the expected accept probability in our method \( \alpha \).

### A.3. Theoretical Predictions vs. Empirical Runtimes

Table 4 compares the expected runtime improvements based on Theorem 3.8 to the empirically measured runtimes from Table 2. We estimated the values of \( c \) for the various models based on profiler traces. We can see that the theoretical predictions mostly match the measured runtimes. The larger differences are due to: (1) optimization differences between our implementation and the baseline, and (2) the simplifying assumption that the \( \beta \)s are i.i.d. being only an approximation (see Section 3.1).

<!-- page 11 -->

### Fast Inference from Transformers via Speculative Decoding

Table 4. Expected improvement factor (EXP) vs. empirically measured improvement factor (EMP).

| TASK | \(M_q\) | TEMP | \(\gamma\) | \(\alpha\) | \(c\) | EXP | EMP |
| :--- | :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| ENDE | T5-SMALL | 0 | 7 | 0.75 | 0.02 | 3.2 | 3.4 |
| ENDE | T5-BASE | 0 | 7 | 0.8 | 0.04 | 3.3 | 2.8 |
| ENDE | T5-LARGE | 0 | 7 | 0.82 | 0.11 | 2.5 | 1.7 |
| ENDE | T5-SMALL | 1 | 7 | 0.62 | 0.02 | 2.3 | 2.6 |
| ENDE | T5-BASE | 1 | 5 | 0.68 | 0.04 | 2.4 | 2.4 |
| ENDE | T5-LARGE | 1 | 3 | 0.71 | 0.11 | 2.0 | 1.4 |
| CNNDM | T5-SMALL | 0 | 5 | 0.65 | 0.02 | 2.4 | 3.1 |
| CNNDM | T5-BASE | 0 | 5 | 0.73 | 0.04 | 2.6 | 3.0 |
| CNNDM | T5-LARGE | 0 | 3 | 0.74 | 0.11 | 2.0 | 2.2 |
| CNNDM | T5-SMALL | 1 | 5 | 0.53 | 0.02 | 1.9 | 2.3 |
| CNNDM | T5-BASE | 1 | 3 | 0.55 | 0.04 | 1.8 | 2.2 |
| CNNDM | T5-LARGE | 1 | 3 | 0.56 | 0.11 | 1.6 | 1.7 |

#### A.4. Application to Beam Search

Our method can be applied, with some performance penalty, to beam search sampling. Given the original beam width \(w\), we can perform beam search with the approximation model \(M_q\) and beam width \(u \ge w\) for \(\gamma\) steps. Then, we can use \(M_p\) to check all of the candidates in parallel (costing a compute budget of \((w + u\gamma)\) runs of \(M_p\)). Finally, for each step, we can accept the guesses of \(M_q\) as long as \(top_w(M_p) \subseteq top_u(M_q)\) to get identical results to regular beam search with \(M_p\) alone (with a more elaborate procedure we could also accept cases where the candidates we got happen to have higher probabilities than those of \(M_p\) alone). The analysis of our method in this setting is more involved and we leave it for future work.

#### A.5. Lenience

A strong property of Algorithm 1 is that the output distribution is guaranteed to remain unchanged. That said, if we're willing to allow some changes, with nice guarantees, we can get further inference speed improvements. To further motivate this, note that when we train two models with identical architectures and sizes on the same dataset, the generated probability distributions will not be identical, so some lenience might make sense. Note that the results in this paper except for this section use the strictest version of Algorithm 1 and don't allow lenience of any kind.

We could include a lenience parameter \(l \in [0, 1]\) and multiply \(q(x)\) by \(l\) before comparing with \(p(x)\) in Algorithm 1. This still maintains the nice guarantee that no token can be sampled with probability greater than \(\frac{p(x)}{l}\). This means for example, that with \(l = \frac{1}{10}\) no token can be sampled with more than 10X its ground truth probability, so we can guarantee that extremely rare tokens will remain extremely rare (there is no guarantee on the minimum probability, so lenience could hurt the diversity of the samples).

Specifically, with a lenience factor \(l\) we have \(\alpha = E_{x \sim q(x)} \begin{cases} 1 & lq(x) \le p(x) \\ \frac{p(x)}{lq(x)} & lq(x) > p(x) \end{cases} = E_{x \sim q(x)} \frac{p(x)}{max(p(x), lq(x))} = \sum_x \frac{p(x)q(x)}{max(p(x), lq(x))} = \frac{1}{l} \sum_x min(p(x), lq(x)) = \sum_x min(\frac{p(x)}{l}, q(x))\).

Table 5 shows \(\alpha\) values for different values of \(l\) when \(M_p\) is T5-XXL (11B) and \(M_q\) is T5-small (77M). With \(c = 0.015\), using lenience values of 1, 0.5, 0.3, and 0.1 (meaning that no token can be sampled with probability greater than 1X, 2X, 3X and 10X of the ground truth) we get improvement factors of 2.5X, 3.1X, 3.6X, and 5X respectively.

Table 5. \(\alpha\) values for various values of \(l\) with standard sampling where \(M_p\) is T5-XXL (11B) on the EnDe translation task.

| \(M_q\) | \(l=1\) | \(l=0.5\) | \(l=0.3\) | \(l=0.1\) |
| :--- | :---: | :---: | :---: | :---: |
| UNIGRAM | 0.07 | 0.1 | 0.11 | 0.16 |
| BIGRAM | 0.19 | 0.23 | 0.25 | 0.32 |
| T5-SMALL (77M) | 0.62 | 0.71 | 0.76 | 0.84 |
| T5-BASE (250M) | 0.68 | 0.8 | 0.83 | 0.90 |

<!-- page 12 -->

Note that when using temperature = 0 (i.e. argmax sampling), we can no longer use lenience as above. Instead, we could allow some lenience before standardizing the distributions. For example, we could accept the token \(x\) sampled from \(M_q\) in case \(p(x) \leq l \cdot max(p)\). In this case, we measure similar empirical increases in \(\alpha\) values to those with temperature = 1. For example, when using lenience values of 1, 0.5, 0.3, and 0.1 for \(M_p\) T5-XXL \(M_q\) T5-small for English-German translation, we get \(\alpha\) values of 0.75, 0.75, 0.8, 0.87. Taking for example \(c = 0.015\) and \(\gamma = 8\) we get speed improvement factors of 3.3X, 3.3X, 3.9X, and 4.9X respectively[^7].

[^7]: In this case, unlike in the standard sampling case shown in Table 5, a lenience factor of 0.5 doesn't improve the speed-up.

</div>

In [ ]:
doc(aai_coding.write_prose)

# module aai_coding.write_prose:

"""How to write prose that doesn't read as AI slop: read before writing anything for human readers.

# Writing Prose That Doesn't Sound Like AI

Guidelines for writing clear, human-sounding prose. Based on the [Anti-Slop Reference](https://github.com/NousResearch/autonovel/blob/master/ANTI-SLOP.md). Apply these when writing documentation, blog posts, READMEs, or any prose.

Here is a passage from The Economist. Its house style is the register most prose this skill covers should default to: documentation, explanation, argument.

> No form of emissions reduction, though, can quickly bend the current trajectory.[1][2] Endurance is what remains.[3][14] Making air conditioning more efficient, cheap and widespread[10] saves lives[6] and fits well with other policy goals, like making clean electricity cheap to produce and consume.[7] Ten years ago a commitment to phase out the fluorinated gases in cooling systems was reached in Kigali, the capital of Rwanda.[

Speculative decoding was introduced in a 2023 ICML paper.  It is now commonly used.

## Why standard decoding is slow

Autoregressive models generate text one token at a time. Each token depends on all the tokens before it, so decoding K tokens takes K serial runs of the model. You cannot compute token 5 until token 4 exists.

To process tokens, the model must:
1. Read all its weights from memory (the memory-bandwidth cost)
2. Do the arithmetic on those weights (the compute cost)

The bottleneck is memory-bandwidth. The arithmetic itself is comparatively cheap. 

## The core idea of speculative decoding

The weights only get loaded from memory a single time per run, whether you push 1 token or 8 tokens through. Reading the weights once lets you do the arithmetic for many tokens at nearly the same cost as for one token. Standard decoding can't take advantage of this; the serial dependency means there is never more than one new token ready to process per sequence. In speculative decoding, a small, cheap model guesses the next several tokens. The large model can evaluate every position at once, in a single run, loading its weights once instead of once per token.

The large model's parallel run does double duty: it is both the check on the guesses and the source of the real answer. The guesses are checked left to right and accepted up to the first mistake. At that point the large model supplies the correct token, and everything after the rejection is thrown away, since those guesses were built on a wrong token.

It might help to view this with a visualization.  I built the following visualization using the "How to Solve It" method, a JupyUvi server, and HTMX. Please run the following lines of code, or you can view a hosted version here (TODO). You can walk through the steps using the forward button.

## The Visualization

Notice that the speculative decoding line alternates between steps of *$M_q$ guessing* ($M_q$ is the small, fast model) and *$M_p$ verifying* ($M_p$ is the large, slow model).  Since speculative decoding gets 1-5 tokens right at a time, it moves ahead much quicker than the standard decoding model.

This visualization was inspired very directly from Figure 1 of the paper.  In addition to the animation, there are a few differences:

- this shows the standard model above it to offer a contrast
- this shows all the tokens that were guessed and rejected, whereas the paper just shows up until the first rejected guess

## How we built the speculative decoding demo

The demo animates Figure 1 of Leviathan et al. (2023): a small model guesses tokens, the large model verifies them in one parallel run. Accepted guesses appear in green, the first rejection in red, and its correction in blue. We added one thing the paper's figure leaves out: the discarded guesses after the first rejection. They were generated, so we show them.

We built it in FastHTML inside this dialog. `JupyUvi` runs the server in a background thread and `HTMX()` shows the app in an iframe, so we could redefine routes and see changes without restarting anything.

The animation data is pre-scripted.

Key design decisions:

- **Server-driven steps over client-side JS.** Each step is an HTMX request that swaps in the next state. This mirrors the algorithm itself: one round-trip per run of the target model.
- **Stateless routes.** The tick number lives in the URL, not on the server. Any request fully describes what to render, which made the back button free.
- **One container for both tracks.** The standard and speculative tracks must advance on the same clock, so a single `/step` route renders both and swaps them together. Separate self-pulling rows could drift apart.
- **A shared clock of two ticks per model run.** Even ticks show the small model guessing, odd ticks show the verification. The standard track reveals one token per odd tick, at the same pace throughout. Keeping that pace matters: the payoff of the demo is watching the speculative track finish while the standard track is still three tokens in.
- **Buttons over a timer.** We started with timed self-advancing rows, then switched to Back and Next buttons stepping one tick at a time, so a viewer can sit on the verification beat as long as they like.
- **The paper's colors.** Black for committed tokens, grey for guesses, green/red/blue for accepted/rejected/corrected, matching Figure 1 so the demo and the paper read as one artifact.


## Guarantee on performance

Each step therefore yields a variable number of tokens: many when the text is predictable and the small model guesses well, just one when it guesses badly. Even in the worst case, every step produces at least one token, so the method is never slower than standard decoding in serial runs.

Two properties make this more than an approximation trick. The output distribution is provably identical to running the large model alone, thanks to the acceptance rule (speculative sampling). And nothing about the models changes: no retraining, no new architecture. The cost is extra total computation for the discarded guesses, which is why the method helps only when memory bandwidth, not compute, is the bottleneck.

## Understanding the ideas driving speculative decoding

I am confused.  Do you still have to run the more expensive model to determine if the tokens will be accepted?  And if so, how does it save bandwidth?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Excellent question — this is exactly the right thing to be confused about, and getting it clear is the key to the whole paper!

You're right on the first part: **yes, you still run the expensive target model $M_p$** to check every token. Speculative decoding never skips $M_p$. So where does the saving come from?

The trick is that $M_p$ is run on all $\gamma$ guessed tokens **in parallel, in a single run**, instead of one at a time across many serial runs.

Here's the crucial distinction, connecting to your earlier question about the bottleneck. To process tokens, the model must:
1. Read all its weights from memory (the memory-bandwidth cost)
2. Do the arithmetic on those weights (the compute cost)

The insight from your notes is that step 1 dominates — inference is bottlenecked on memory bandwidth, not arithmetic. And here's the key fact about the hardware: **reading the weights once lets you do the math for many tokens at nearly the same cost as for one token.** The weights only get loaded from memory a single time per run, whether you push 1 token or 8 tokens through.

So compare:
- **Standard**: 8 tokens → 8 serial runs → weights loaded **8 times**
- **Speculative**: 8 guessed tokens → 1 parallel run → weights loaded **once**

Before I connect this back to the "serial dependency" problem — does the idea that *loading the weights, not the arithmetic, is the expensive part* make sense to you?

Yes, the idea that loading the weights, not the arithmetic is the expensive part, makes sense. I can see how running the model on multiple tokens in parallel doesn't take longer.  My question is that I still don't see how the speculative runs help us? Don't we have to do a standard run to determine which is correct?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You've hit the exact crux — let me resolve it.

The key realization: **the single parallel run of $M_p$ _is_ the verification.** There's no separate standard run afterward. That one parallel run does double duty.

Remember *why* we normally can't parallelize: to run $M_p$ on token 5, we need token 4 first (the serial dependency). That's the whole bottleneck.

But now we have *guesses* for tokens 1 through 8 from the cheap model $M_q$. So we can hand $M_p$ all these prefixes **at once**:
- $M_p(\text{prefix})$ → its opinion on token 1
- $M_p(\text{prefix} + [x_1])$ → its opinion on token 2
- $M_p(\text{prefix} + [x_1, x_2])$ → its opinion on token 3
- ... and so on

The guesses break the dependency chain — we no longer have to *wait* for each real token, because we've filled in the gaps with guesses. That lets all these evaluations happen in one parallel run, which loads the weights just once.

Then we compare $M_p$'s opinions to the guesses and keep the ones that match (that's the acceptance step).

So to directly answer you: **we never do a separate standard run.** The parallel run replaces it.

Does that clear up the confusion — that the verification and the "standard run" are the same single step?

So the verification run can check all steps in a sequence at once?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Exactly right! That's the heart of it. One parallel run of $M_p$ checks all $\gamma$ guessed steps simultaneously, because we supplied the guessed tokens to fill in each prefix.

There's one subtlety worth making sure of. The checking happens **left to right**, and we accept guesses only up to the *first* mistake. So if $M_p$ disagrees with, say, the guess at position 3, then:
- guesses 1 and 2 are accepted ✓
- guess 3 is rejected, and we use $M_p$'s own answer there instead ✗
- guesses 4 onward are thrown away — even if some happened to be "right"

Why must we discard everything after the first rejection, do you think? (Hint: it comes back to the serial dependency we just talked about.)

It seems like speculative decoding has successfully separated generation from validation.  Why were generation and validation more closely coupled together prior to this)?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

That's a beautifully framed question — and the answer reveals *why* the whole idea is clever.

In standard autoregressive decoding, generation and validation weren't just *coupled* — they were literally **the same act**. When $M_p$ runs at position $t$, the token it produces *is* the answer. There's nothing to validate, because the act of generating and the act of deciding-what's-correct are one and the same operation, done by one model.

So the "coupling" came from using **a single model** to do both jobs. That single model is the authority on what's correct, so whatever it generates is, by definition, accepted.

Speculative decoding breaks this apart by introducing a **second, cheaper model**. Now the roles can be split:
- $M_q$ (cheap) → *proposes* candidate tokens
- $M_p$ (expensive) → *validates* them

Here's the question to sit with: this separation only *helps* because of something specific about the hardware cost of $M_p$ that we discussed earlier. Which property of $M_p$ makes it worthwhile to have it *validate a batch* rather than *generate one at a time*?

## Difference from diagrams in the paper

The diagram from the paper only shows the first token to be rejected, but doesn't show other tokens that were generated but discarded.  I think it would be clearer to show all generated tokens.

I have a question.  In this example, they only show up to the first guess that was rejected.  In practice, wouldn't there be more tokens?  I realize that after 1 is rejected, the rest aren't read/verified, but aren't they still generated?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Exactly right — you've understood the algorithm precisely.

$M_q$ generates all $\gamma$ guesses up front, autoregressively, *before* $M_p$ ever looks at them. So yes: the tokens after the first rejection **were** generated — they're just thrown away once $M_p$'s verdict comes in.

So for a given iteration with $\gamma = 8$ guesses and a rejection at position 3, we'd have:

- 8 grey guesses proposed by $M_q$
- $M_p$ validates all 8 in parallel
- positions 1–2 accepted, position 3 rejected + corrected
- positions 4–8: **generated but discarded**

Your data captures only the *kept* tokens, which is what the *output* looks like — but the *discarded* guesses are part of the real work, and (as you noticed earlier) they're the reason total arithmetic goes *up* even as walltime goes down.

This is actually a nice opportunity: should our animation **show those discarded guesses** — propose all $\gamma$ in grey, then visibly fade/drop the ones after the first rejection? That would make the "wasted speculation" visible, which is pedagogically valuable.

If you'd like that, we'd want to extend the data so each iteration records the *full* list of guesses (including the doomed ones), not just the accepted prefix. Want to go that route?

## Plan for a Visualization

Can you think of a visualization that would illustrate / animate this process clearly?  For someone to see strings of tokens being generated, and validated or rejected?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes! Here's a visualization concept that would map naturally onto how the algorithm actually works.

The core idea: an **animated grid**, one row per iteration, mirroring Figure 1 but *in motion*.

Each iteration animates in three beats:

1. **Proposal** — the cheap model $M_q$ types out $\gamma$ guessed tokens left-to-right, shown in a neutral/grey "tentative" color (maybe with a slight flicker to signal "these are just guesses").

2. **Validation** — $M_p$'s single parallel run fires. To emphasize that this is *one simultaneous step*, all the guessed tokens get checked **at once** — e.g. a sweep or flash across the whole row, rather than a left-to-right crawl.

3. **Resolution** — the tokens recolor: accepted guesses turn **green**, the first rejected one turns **red**, and its correction appears in **blue**. Everything after the red token visibly **fades/drops away** (discarded).

Then the next row begins, continuing from the accepted tokens plus the correction.

Two touches that would really drive the key insights home:
- A **counter** showing "tokens generated this step" (1 to $\gamma+1$), so the *variable* yield you noticed becomes visible.
- A side-by-side **"standard decoding" track** running in real-time alongside, so you can watch speculative decoding pull ahead — making the walltime speedup tangible.

Before I go further — which aspect matters most to you to convey: the **parallelism** (that validation is one simultaneous step), the **accept/reject mechanics**, or the **speedup** versus standard decoding? That'll shape which details to emphasize.

I like your idea.  I think that the parallelism, accept/reject, and speedup are all important attributes.  Could you help me build this demo using fasthtml components, inside this SolveIt dialog?  please take time to get familiar with FastHTML first using these docs: https://www.fastht.ml/docs/ 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me start by reading the FastHTML docs to get familiar with it.



<details class='tool-usage-details' markdown='1'>
<summary><code>read_url(url=&quot;https://www.fastht.ml/docs/&quot;)→&quot;[ ![](https://www.fastht.ml/docs/logo.sv…&quot;</code></summary>

```json
{
  "id": "toolu_01UnXu9RWVc3yCiqx3aeFQBq",
  "server": false,
  "call": {
    "function": "read_url",
    "arguments": {
      "url": "https://www.fastht.ml/docs/"
    }
  },
  "result": "<TRUNCATED>…s://www.fastht.ml/docs/logo.svg) ](./index.html)\n\n  * [ Home](https://fastht.ml)\n  * [ Learn](https://fastht.ml/about)\n\n\n  * [ __ ](https://github.com/answerdotai/fasthtml)\n  * [ __ ](https://x.com/answerdotai)\n\n\n\n__\n\n  1. [Get Started](./index.html)\n\n\n\n  * [ Get Started](./index.html)\n\n  * [ Tutorials](./tutorials/index.html) __\n\n    * [ FastHTML By Example](./tutorials/by_example.html)\n\n    * [ Web Devs Quickstart](./tutorials/quickstart_for_web_devs.html)\n\n    * [ JS App Walkthrough](./tutorials/e2e.html)\n\n    * [ Using Jupyter to write FastHTML](./tutorials/jupyter_and_fasthtml.html)\n\n  * Explanations __\n\n    * [ Background Tasks](./explains/background_tasks.html)\n\n    * [ **FT** Components](./explains/explaining_xt_components.html)\n\n    * [ FAQ](./explains/faq.html)\n\n    * [ MiniDataAPI Spec](./explains/minidataapi.html)\n\n    * [ OAuth](./explains/oauth.html)\n\n    * [ Routes](./explains/routes.html)\n\n    * [ Stripe](./explains/stripe.html)\n\n    * [ WebSockets](./explains/websockets.html)\n\n  * Reference __\n\n    * [ Concise reference](./ref/concise_guide.html)\n\n    * [ FastHTML Best Practices](./ref/best_practice.html)\n\n    * [ Custom Components](./ref/defining_xt_component.html)\n\n    * [ Handling handlers](./ref/handlers.html)\n\n    * [ Live Reloading](./ref/live_reload.html)\n\n    * [ Response Types](./ref/response_types.html)\n\n  * Source __\n\n    * [ Core](./api/core.html)\n\n    * [ Components](./api/components.html)\n\n    * [ Component extensions](./api/xtend.html)\n\n    * [ Javascript examples](./api/js.html)\n\n    * [ Pico.css components](./api/pico.html)\n\n    * [ SVG](./api/svg.html)\n\n    * [ Jupyter compatibility](./api/jupyter.html)\n\n    * [ OAuth](./api/oauth.html)\n\n    * [ Command Line Tools](./api/cli.html)\n\n\n\n\n## On this page\n\n  * Installation\n  * Usage\n    * Getting help from AI\n  * Next Steps\n  * Other languages and related projects\n\n\n\n  * [__Report an issue](https://github.com/AnswerDotAI/fasthtml/issues/new)\n\n\n\n## Other Formats\n\n  * [ __CommonMark](index.html.md)\n\n\n\n# FastHTML\n\nThe fastest, most powerful way to create an HTML app \n\nWelcome to the official FastHTML documentation.\n\nFastHTML is a new next-generation web framework for fast, scalable web applications with minimal, compact code. It’s designed to be:\n\n  * Powerful and expressive enough to build the most advanced, interactive web apps you can imagine.\n  * Fast and lightweight, so you can write less code and get more done.\n  * Easy to learn and use, with a simple, intuitive syntax that makes it easy to build complex apps quickly.\n\n\n\nFastHTML apps are just Python code, so you can use FastHTML with the full power of the Python language and ecosystem. FastHTML’s functionality maps 1:1 directly to HTML and HTTP, but allows them to be encapsulated using good software engineering practices—so you’ll need to understand these foundations to use this library fully. To understand how and why this works, please read this first: [fastht.ml/about](https://fastht.ml/about).\n\n## Installation\n\nSince `fasthtml` is a Python library, you can install it with:\n````\npip install python-fasthtml __\n````\n\nIn the near future, we hope to add component libraries that can likewise be installed via `pip`.\n\n## Usage\n\nFor a minimal app, create a file “main.py” as follows:\n````\n**main.py**\n````\n````\nfrom fasthtml.common import *\n\napp,rt = fast_app()\n\n@rt('/')\ndef get(): return Div(P('Hello World!'), hx_get=\"/change\")\n\nserve()__\n````\n\nRunning the app with `python main.py` prints out a link to your running app: `http://localhost:5001`. Visit that link in your browser and you should see a page with the text “Hello World!”. Congratulations, you’ve just created your first FastHTML app!\n\nAdding interactivity is surprisingly easy, thanks to HTMX. Modify the file to add this function:\n````\n**main.py**\n````\n````\n@rt('/change')\ndef get(): return P('Nice to be here!')__\n````\n\nYou now have a page with a clickable element that changes the text when clicked. When clicking on this link, the server will respond with an “HTML partial”—that is, just a snippet of HTML which will be inserted into the existing page. In this case, the returned element will replace the original P element (since that’s the default behavior of HTMX) with the new version returned by the second route.\n\nThis “hypermedia-based” approach to web development is a powerful way to build web applications.\n\n### Getting help from AI\n\nBecause FastHTML is newer than most LLMs, AI systems like Cursor, ChatGPT, Claude, and Copilot won’t give useful answers about it. To fix that problem, we’ve provided an LLM-friendly guide that teaches them how to use FastHTML. To use it, add this link for your AI helper to use:\n\n  * [/llms-ctx.txt](https://www.fastht.ml/docs/llms-ctx.txt)\n\n\n\nThis example is in a format based on recommendations from Anthropic for use with [Claude Projects](https://support.anthropic.com/en/articles/9517075-what-are-projects). This works so well that we’ve actually found that Claude can prov…</TRUNCATED>"
}
```

</details>





<details class='tool-usage-details' markdown='1'>
<summary><code>read_url(url=&quot;https://www.fastht.ml/docs/tutorials/jup…&quot;)→&quot;[ ![](https://www.fastht.ml/docs/logo.sv…&quot;</code></summary>

```json
{
  "id": "toolu_01Uw2zEcomqPLciGSEymKwCG",
  "server": false,
  "call": {
    "function": "read_url",
    "arguments": {
      "url": "https://www.fastht.ml/docs/tutorials/jupyter_and_fasthtml.html"
    }
  },
  "result": "<TRUNCATED>…s://www.fastht.ml/docs/logo.svg) ](../index.html)\n\n  * [ Home](https://fastht.ml)\n  * [ Learn](https://fastht.ml/about)\n\n\n  * [ __ ](https://github.com/answerdotai/fasthtml)\n  * [ __ ](https://x.com/answerdotai)\n\n\n\n__\n\n  1. [Tutorials](../tutorials/index.html)\n  2. [Using Jupyter to write FastHTML](../tutorials/jupyter_and_fasthtml.html)\n\n\n\n  * [ Get Started](../index.html)\n\n  * [ Tutorials](../tutorials/index.html) __\n\n    * [ FastHTML By Example](../tutorials/by_example.html)\n\n    * [ Web Devs Quickstart](../tutorials/quickstart_for_web_devs.html)\n\n    * [ JS App Walkthrough](../tutorials/e2e.html)\n\n    * [ Using Jupyter to write FastHTML](../tutorials/jupyter_and_fasthtml.html)\n\n  * Explanations __\n\n    * [ Background Tasks](../explains/background_tasks.html)\n\n    * [ **FT** Components](../explains/explaining_xt_components.html)\n\n    * [ FAQ](../explains/faq.html)\n\n    * [ MiniDataAPI Spec](../explains/minidataapi.html)\n\n    * [ OAuth](../explains/oauth.html)\n\n    * [ Routes](../explains/routes.html)\n\n    * [ Stripe](../explains/stripe.html)\n\n    * [ WebSockets](../explains/websockets.html)\n\n  * Reference __\n\n    * [ Concise reference](../ref/concise_guide.html)\n\n    * [ FastHTML Best Practices](../ref/best_practice.html)\n\n    * [ Custom Components](../ref/defining_xt_component.html)\n\n    * [ Handling handlers](../ref/handlers.html)\n\n    * [ Live Reloading](../ref/live_reload.html)\n\n    * [ Response Types](../ref/response_types.html)\n\n  * Source __\n\n    * [ Core](../api/core.html)\n\n    * [ Components](../api/components.html)\n\n    * [ Component extensions](../api/xtend.html)\n\n    * [ Javascript examples](../api/js.html)\n\n    * [ Pico.css components](../api/pico.html)\n\n    * [ SVG](../api/svg.html)\n\n    * [ Jupyter compatibility](../api/jupyter.html)\n\n    * [ OAuth](../api/oauth.html)\n\n    * [ Command Line Tools](../api/cli.html)\n\n\n\n\n## On this page\n\n  * Full screen view\n  * Graceful shutdowns\n\n\n\n  * [__Report an issue](https://github.com/AnswerDotAI/fasthtml/issues/new)\n\n\n\n## Other Formats\n\n  * [ __CommonMark](jupyter_and_fasthtml.html.md)\n\n\n\n  1. [Tutorials](../tutorials/index.html)\n  2. [Using Jupyter to write FastHTML](../tutorials/jupyter_and_fasthtml.html)\n\n\n\n# Using Jupyter to write FastHTML\n\nWriting FastHTML applications in Jupyter notebooks requires a slightly different process than normal Python applications. \n\nWriting FastHTML applications in Jupyter notebooks requires a slightly different process than normal Python applications.\n\n__\n\nDownload this notebook and try it yourself \n\nThe source code for this page is a [Jupyter notebook](https://github.com/AnswerDotAI/fasthtml/blob/main/nbs/tutorials/jupyter_and_fasthtml.ipynb). That makes it easy to directly experiment with it. However, as this is working code that means we have to comment out a few things in order for the documentation to build.\n\nThe first step is to import necessary libraries. As using FastHTML inside a Jupyter notebook is a special case, it remains a special import.\n````\nfrom fasthtml.common import *\nfrom fasthtml.jupyter import JupyUvi, HTMX __\n````\n\nLet’s create an app with `fast_app`.\n````\napp, rt = fast_app(pico=True)__\n````\n\nDefine a route to test the application.\n````\n@rt\ndef index():\n    return Titled('Hello, Jupyter',\n           P('Welcome to the FastHTML + Jupyter example'),\n           Button('Click', hx_get='/click', hx_target='#dest'),\n           Div(id='dest')\n    )__\n````\n\nCreate a `server` object using [`JupyUvi`](https://www.fastht.ml/docs/api/jupyter.html#jupyuvi), which also starts Uvicorn. The `server` runs in a separate thread from Jupyter, so it can use normal HTTP client functions in a notebook.\n````\nserver = JupyUvi(app)__\n````\n\nThe [`HTMX`](https://www.fastht.ml/docs/api/jupyter.html#htmx) callable displays the server’s HTMX application in an iframe which can be displayed by Jupyter notebook. Pass in the same `port` variable used in the [`JupyUvi`](https://www.fastht.ml/docs/api/jupyter.html#jupyuvi) callable above or leave it blank to use the default (8000).\n````\n# This doesn't display in the docs - uncomment and run it to see it in action\n# HTMX()__\n````\n\nWe didn’t define the `/click` route, but that’s fine - we can define (or change) it any time, and it’s dynamically inserted into the running app. No need to restart or reload anything!\n````\n@rt\ndef click(): return P('You clicked me!')__\n````\n\n## Full screen view\n\nYou can view your app outside of Jupyter by going to `localhost:PORT`, where `PORT` is usually the default 8000, so in most cases just click [this link](localhost:8000/).\n\n## Graceful shutdowns\n\nUse the `server.stop()` function displayed below. If you restart Jupyter without calling this line the thread may not be released and the [`HTMX`](https://www.fastht.ml/docs/api/jupyter.html#htmx) callable above may throw errors. If that happens, a quick temporary fix is to specify a different port number in JupyUvi and HTMX with the `port` parameter.\n\nCleaner solutions to the dangling…</TRUNCATED>"
}
```

</details>

I've read the FastHTML getting-started docs and, importantly, the **"Using Jupyter to write FastHTML"** tutorial — that one's key for us, since a Solveit dialog works like a notebook. Here's what I learned that matters for our demo:

- FastHTML apps are plain Python: `app, rt = fast_app()`, then routes via `@rt`, returning components like `Div`, `P`, `Button`.
- Inside a notebook/dialog we use `from fasthtml.jupyter import JupyUvi, HTMX`: `JupyUvi(app)` runs the server in a background thread, and `HTMX()` displays the app in an iframe right in the dialog.
- Routes can be added or redefined *live* — no server restart needed. Perfect for building up our animation step by step.
- Interactivity comes via HTMX attributes (e.g. `hx_get`, `hx_target`), though for smooth animation we may add a sprinkle of CSS/JS.

Here's the plan I'd suggest, in small steps:

1. **Get a minimal FastHTML app running** in the dialog (server + iframe showing "hello").
2. **Render one static iteration row** — tokens as colored spans (grey guesses, then green/red/blue) with simple CSS.
3. **Animate one iteration** — the three beats: propose → validate (flash) → resolve (recolor + fade discarded).
4. **Run whole sequence** — multiple rows using pre-scripted token data (e.g. the Nikkei sentence), with the per-step token counter.
5. **Add the side-by-side standard-decoding track** for the speedup comparison.


## Code

### imports and constants

In [ ]:
!pip install python-fasthtml

In [ ]:
from fasthtml.common import *
from fasthtml.jupyter import JupyUvi, HTMX

In [ ]:
colors = {'accepted':'#22c55e', 'rejected':'#ef4444', 'correction':'#3b82f6', 'committed':'#000000', 'guess':'#9ca3af'}

In [ ]:
all_sentences = [
    ({'prefix': [], 'guesses': ['Global', 'stocks', 'fell', 'sharply', 'after'],
     'n_accepted': 1, 'correction': 'markets'},
    {'prefix': ['Global', 'markets'],
     'guesses': ['surged', 'after', 'the', 'central', 'government', 'decided'],
     'n_accepted': 4, 'correction': 'bank'},
    {'prefix': ['Global', 'markets', 'surged', 'after', 'the', 'central', 'bank'],
     'guesses': ['announced', 'unexpected', 'interest', 'rate', 'changes'],
     'n_accepted': 2, 'correction': 'rate'},
    {'prefix': ['Global', 'markets', 'surged', 'after', 'the', 'central', 'bank', 'announced', 'unexpected', 'rate'],
     'guesses': ['cuts', 'last', 'week', 'following', 'pressure'],
     'n_accepted': 2, 'correction': 'Friday'},
    ['Global', 'markets', 'surged', 'after', 'the', 'central', 'bank', 'announced', 'unexpected', 'rate', 'cuts', 'last', 'Friday']),
    ({'prefix': [], 'guesses': ['The', 'robot', 'quickly', 'started', 'to', 'move'],
     'n_accepted': 2, 'correction': 'slowly'},
    {'prefix': ['The', 'robot', 'slowly'],
     'guesses': ['learned', 'to', 'adapt', 'through', 'trial'],
     'n_accepted': 2, 'correction': 'paint'},
    {'prefix': ['The', 'robot', 'slowly', 'learned', 'to', 'paint'],
     'guesses': ['pictures', 'of', 'the', 'sea', 'and', 'sky'],
     'n_accepted': 3, 'correction': 'ocean'},
    ['The', 'robot', 'slowly', 'learned', 'to', 'paint', 'pictures', 'of', 'the', 'ocean']),  
    ({'prefix': [], 'guesses': ['The', 'new', 'telescope', 'found'],
     'n_accepted': 1, 'correction': 'satellite'},
    {'prefix': ['The', 'satellite'],
     'guesses': ['captured', 'detailed', 'images', 'of', 'the', 'surface'],
     'n_accepted': 4, 'correction': 'volcanic'},
    {'prefix': ['The', 'satellite', 'captured', 'detailed', 'images', 'of', 'volcanic'],
     'guesses': ['activity', 'deep', 'below', 'ground'],
     'n_accepted': 2, 'correction': 'beneath'},
    {'prefix': ['The', 'satellite', 'captured', 'detailed', 'images', 'of', 'volcanic', 'activity', 'deep', 'beneath'],
     'guesses': ['the', 'southern', 'polar', 'cap', 'region'],
     'n_accepted': 2, 'correction': 'ice'},
    {'prefix': ['The', 'satellite', 'captured', 'detailed', 'images', 'of', 'volcanic', 'activity', 'deep', 'beneath', 'the', 'southern', 'ice'],
     'guesses': ['caps', 'melting', 'rapidly'],
     'n_accepted': 0, 'correction': 'shelf'},
    ['The', 'satellite', 'captured', 'detailed', 'images', 'of', 'volcanic', 'activity', 'deep', 'beneath', 'the', 'southern', 'ice', 'shelf']),
    ({'prefix': [], 'guesses': ['Once', 'upon', 'a', 'time', 'there', 'was'],
     'n_accepted': 5, 'correction': 'lived'},
    {'prefix': ['Once', 'upon', 'a', 'time', 'there', 'lived'],
     'guesses': ['a', 'brave', 'little', 'girl', 'who', 'loved'],
     'n_accepted': 3, 'correction': 'rabbit'},
    {'prefix': ['Once', 'upon', 'a', 'time', 'there', 'lived', 'a', 'brave', 'little', 'rabbit'],
     'guesses': ['in', 'a', 'cozy', 'hole', 'under'],
     'n_accepted': 3, 'correction': 'burrow'},
    ['Once', 'upon', 'a', 'time', 'there', 'lived', 'a', 'brave', 'little', 'rabbit', 'in', 'a', 'cozy', 'burrow'])
]

### App: Version 3 (Standard version + Speculative Decoding + Multiple Sentences)

In [ ]:
# the app only needs to be created once
app, rt = fast_app(pico=True)

In [ ]:
def Tok(text, status, extra=''):
    return Span(text, style=f'background:{colors[status]};color:white;padding:2px 6px;margin:2px;border-radius:4px;{extra}')

In [ ]:
def tick_state(tick, iters):
    n_iters = len(iters)
    spec_iter = tick // 2           # which speculative iteration
    is_resolve = tick % 2 == 1      # propose or resolve?
    spec_done = spec_iter >= n_iters
    n_standard = (tick + 1) // 2    # standard reveals one token per odd tick
    return spec_iter, is_resolve, spec_done, n_standard

In [ ]:
def wrap_row(name, row):
    return Div(Strong(name, style='width:8em'), row, style='display:flex;align-items:center')

In [ ]:
@rt
def step(tick: int = 0, sent: int=0):
    *iters, std_tokens = all_sentences[sent]
    spec_iter, is_resolve, spec_done, n_standard = tick_state(tick, iters)
    
    # determine phase for speculative decoding
    phase = Span('M', Sub('p'), 'verifying (parallel)') if is_resolve else Span('M', Sub('q'), ' guessing')
    if spec_done: phase = 'finished'

    # standard approach (without speculative decoding)
    if n_standard == 0: std_row = Div(Em('(running…)', style='color:#9ca3af'))
    else: std_row = Div(Tok(t, 'committed') for t in std_tokens[:n_standard])

    if spec_done: 
        spec_row = Div(*(Tok(t, 'committed') for t in std_tokens), Span(phase, style='color:#9ca3af;margin-left:1em'))
    else:
        it = iters[spec_iter]
        prefix = [Tok(t, 'committed') for t in it['prefix']]
        if is_resolve:
            g = it['guesses']
            green = [Tok(t, 'accepted') for t in g[:it['n_accepted']]]      # accepted                       
            red = Tok(g[it['n_accepted']], 'rejected', 'text-decoration:line-through')      # first rejection
            blue = Tok(it['correction'], 'correction')                      # M_p's fix
            # g[n_accepted+1:] are discarded — not rendered
            spec_row = Div(*prefix, *green, red, blue, Span(phase, style='color:#9ca3af;margin-left:1em'))
        else:
            guesses = [Tok(t, 'guess') for t in it['guesses']]
            spec_row = Div(*prefix, *guesses, Span(phase, style='color:#9ca3af;margin-left:1em'))

    counter = Span(f'Step {tick//2 + 1} of {len(std_tokens)}', style='margin:0 1em')
    back_btn = Button('◀ Back', hx_get=step.to(tick=tick-1, sent=sent), hx_target='#tracks', hx_swap='outerHTML', disabled=tick==0)
    next_btn = Button('Next ▶', hx_get=step.to(tick=tick+1, sent=sent), hx_target='#tracks', hx_swap='outerHTML', 
                disabled=tick+1>=2*len(std_tokens))
    
    change_btn = Button('Change sentence', hx_get=step.to(tick=0, sent=(sent+1) % len(all_sentences)), hx_target='#tracks',    
                hx_swap='outerHTML')   

    return Div(wrap_row('Standard', std_row), Hr(), wrap_row('Speculative Decoding', spec_row), Hr(),
           Div(back_btn, counter, next_btn), Hr(), Div(change_btn), id='tracks', hx_swap='outerHTML')



In [ ]:
@rt
def index(): return Titled('Speculative Decoding Visualization', step(0))

In [ ]:
# keep the server = JupyUvi(app) line in its own cell, separate from your route definitions — then you can re-run the routes freely without touching the server

server = JupyUvi(app)

In [ ]:
HTMX(height=350)

HTML(<iframe src="http://localhost:8000/" style="width: 100%; height: 350px; border: none;" onload="" allow="accelerometer; autoplay; camera; clipboard-read; clipboard-write; display-capture; encrypted-media; fullscreen; gamepad; geolocation; gyroscope; hid; identity-credentials-get; idle-detection; magnetometer; microphone; midi; payment; picture-in-picture; publickey-credentials-get; screen-wake-lock; serial; usb; web-share; xr-spatial-tracking"></iframe> )

flexbox froggy -- can help with how this is running off the screen

In [ ]:
server.stop()

## Coding choices

Let's go with the server-driven HTMX approach then.  I like that it embodies the algorithm and that it is more idiomatic.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great choice — let's design the HTMX flow before writing code.

The core mechanism in HTMX for a timed sequence is a `Div` that requests its own next state from the server, using `hx_trigger` with a delay. So each beat is a server response that swaps in the row's new appearance, and includes the trigger for the *next* beat.

For one iteration, the three server responses would be:

1. **Propose** — render $M_q$'s guesses as grey tokens; include `hx_trigger="load delay:1s"` to fetch beat 2.
2. **Validate** — same tokens but with a "flash" style (signalling $M_p$'s parallel run); trigger beat 3 after a short delay.
3. **Resolve** — recolor to green/red/blue and drop the discarded tokens; then trigger the *next iteration*.

So the row essentially "pulls" itself through the beats, each swap being one server round-trip — which, as we said, mirrors a run of $M_p$.

One design question to settle first: should the server hold the **state** (which iteration/beat we're on) so each request just says "give me the next step"? Or should each request carry the state in its URL/parameters (e.g. `/step?iter=0&beat=1`)?

The second (stateless, params-in-URL) is usually cleaner and easier to reason about. Which do you prefer?

Let's got with stateless, params-in-URL

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good — stateless keeps the logic transparent, and it means any request fully describes what to render, with no hidden server memory to get out of sync.

Before wiring up routes, we need the **data** the routes will read: the pre-scripted sequence of iterations. Since the server is just a "playback" of a fixed run here, each iteration needs to know:

- the **committed prefix** (black tokens carried from before)
- the **guesses** $M_q$ proposed this iteration
- **how many** were accepted before the first rejection
- the **correction** token from $M_p$

A natural shape is a list of iterations, each something like:

```python
{'prefix': [...committed tokens...],
 'guesses': [...γ guessed tokens...],
 'n_accepted': 4,          # accepted before first rejection
 'correction': 'index'}    # M_p's replacement for the rejected guess
```

From that one record, a route can derive every beat's colors: propose = all guesses grey; resolve = first `n_accepted` green, next one red, correction blue, rest dropped.

Does this data shape look right to you? If so, the next small step is for you to write out two or three iterations of the Nikkei sentence in this form, and we'll build the first route against it.

What is up with the .to syntax used by HTMX?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

That's a FastHTML feature, not HTMX itself. When you decorate a function with `@rt`, FastHTML attaches a `.to()` method to it that generates the URL for that route, including query parameters. So:

```python
resolve.to(i=2)
```

produces something like `/resolve?i=2`.

It's FastHTML's equivalent of Flask's `url_for()` — you reference the function directly instead of hardcoding URL strings, so if you rename a route, everything stays in sync.

remind me what OuterHTML does?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

In HTMX, `hx_swap='outerHTML'` means the server's response **replaces the entire element** that made the request — including the element itself, not just its children.

Compare the two main options:
- **`innerHTML`** (the default) — replaces only the *contents inside* the element; the element's own tag stays.
- **`outerHTML`** — replaces the *whole element*, tag and all, with whatever the server returns.

We need `outerHTML` here because each beat returns a complete new `Div(id='row', ...)` that should fully swap out the previous one — keeping the same `id` so the next beat can target it again.

I'd like the phase label attached to the speculative row.  Reviewing the paper again, what are the clearest and most accurate choices of labels for the 2 phases?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Looking back at Algorithm 1, the paper's own comments for the two phases are:

1. *"Sample γ guesses from $M_q$ autoregressively"*
2. *"Run $M_p$ in parallel"* (then accept/reject via speculative sampling)

So a couple of label options, from most paper-faithful to most plain-English:

- **"$M_q$ guessing"** / **"$M_p$ verifying (parallel)"** — closest to the paper's language, and nicely reinforces *which model* is doing the work in each phase
- **"Drafting"** / **"Verification"** — the terminology much of the follow-up literature settled on
- **"Speculation"** / **"Validation"** — your original suggestion, also fine

One note: "Speculation vs. validation" is slightly loose in the paper's terms, since the *whole method* is speculative — the second phase includes both the parallel $M_p$ run *and* the accept/reject decisions.

My lean would be option 1, since showing the model names teaches the reader the mechanics for free. Which do you prefer?

Why doesn't it take longer to have separate generate and validate steps, as opposed to just having a generation step?

Because the "generate" step uses $M_q$, which is *tiny* compared to $M_p$. The paper reports the cost coefficient $c$ (ratio of one $M_q$ run to one $M_p$ run) is typically less than 0.05 — often negligible.

So a speculative step costs roughly: **$\gamma$ cheap runs of $M_q$ + 1 run of $M_p$** ≈ **1 run of $M_p$**.

A standard step costs: **1 run of $M_p$** → produces **1 token**.

I am happy with how the FastHTML app is working now. I am wondering if you can write a short summary of how we made the app and some of the key decisions in its design?  Please use the tips above on writing good prose to make sure that your summary doesn't sound like slop.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here's a draft summary you can paste into a note:

```markdown
## How we built the speculative decoding demo

The demo animates Figure 1 of Leviathan et al. (2023): a small model guesses tokens, the large model verifies them in one parallel run, and accepted guesses appear in green, the first rejection in red, and its correction in blue. We added one thing the paper's figure leaves out: the discarded guesses after the first rejection. They were generated, so we show them.

We built it in FastHTML inside this dialog. `JupyUvi` runs the server in a background thread and `HTMX()` shows the app in an iframe, so we could redefine routes and see changes without restarting anything.

The animation data is pre-scripted. Each iteration is a dict holding the committed prefix, the guesses, how many were accepted, and the correction. Each prefix must equal the previous prefix plus its accepted guesses plus its correction, so the whole run can be checked by hand.

Key design decisions, in the order we hit them:

- **Server-driven steps over client-side JS.** Each step is an HTMX request that swaps in the next state. This mirrors the algorithm itself: one round-trip per run of the target model.
- **Stateless routes.** The tick number lives in the URL, not on the server. Any request fully describes what to render, which made the back button free.
- **One container for both tracks.** The standard and speculative tracks must advance on the same clock, so a single `/step` route renders both and swaps them together. Separate self-pulling rows could drift apart.
- **A shared clock of two ticks per model run.** Even ticks show the small model guessing, odd ticks show the verification. The standard track reveals one token per odd tick, at the same pace throughout. Keeping that pace honest matters: the payoff of the demo is watching the speculative track finish while the standard track is still three tokens in.
- **Buttons over a timer.** We started with timed self-advancing rows, then switched to Back and Next buttons stepping one tick at a time, so a viewer can sit on the verification beat as long as they like.
- **The paper's colors.** Black for committed tokens, grey for guesses, green/red/blue for accepted/rejected/corrected, matching Figure 1 so the demo and the paper read as one artifact.
```

Anything you'd like reworded, cut, or added? For instance, I left out the phase labels and the step counter, which we could mention under the buttons decision.

Can you summarize our earlier discussion about the core underlying ideas of speculative decoding, in a way that would be clear to someone who knows the basics of machine learning but is not familiar with speculative decoding?  Please use good prose, not slop.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here's a draft:

```markdown
## The core ideas of speculative decoding

Autoregressive models generate text one token at a time. Each token depends on all the tokens before it, so decoding K tokens takes K serial runs of the model. You cannot compute token 5 until token 4 exists.

A single run of a large model is slow for a surprising reason. The bottleneck is not arithmetic. It is memory bandwidth: the time spent loading the model's weights from memory. The arithmetic itself is comparatively cheap. This asymmetry means that once the weights are loaded, pushing eight tokens through the model costs about the same as pushing one.

That fact is useless for standard decoding, because the serial dependency means there is never more than one token ready to process. Speculative decoding manufactures the missing tokens. A small, cheap model guesses the next several tokens. Those guesses fill in the gaps in the dependency chain, so the large model can evaluate every position at once, in a single run, loading its weights once instead of once per token.

The large model's parallel run does double duty: it is both the check on the guesses and the source of the real answer. There is no separate "standard" run afterwards. The guesses are checked left to right and accepted up to the first mistake. At that point the large model supplies the correct token, and everything after the rejection is thrown away, since those guesses were built on a wrong token.

Each step therefore yields a variable number of tokens: many when the text is predictable and the small model guesses well, just one when it guesses badly. Even in the worst case, every step produces at least one token, so the method is never slower than standard decoding in serial runs.

Two properties make this more than an approximation trick. The output distribution is provably identical to running the large model alone, thanks to the acceptance rule (speculative sampling). And nothing about the models changes: no retraining, no new architecture. The cost is extra total computation for the discarded guesses, which is why the method helps only when memory bandwidth, not compute, is the bottleneck.
```

Want me to adjust the level of detail anywhere, or trim it down?

I like this more recent summary better.  Can you rewrite it, keeping the same ideas, but make it less slop-like / use better prose?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here's the rewrite:

```markdown
## How we built the speculative decoding visualizer

The app animates the paper's Figure 1: a small model proposes tokens, the large model verifies them in one parallel run, and the viewer watches speculative decoding pull ahead of standard decoding. It runs in FastHTML inside this Solveit dialog. `JupyUvi` runs the server in a background thread and `HTMX()` shows the app in an iframe, so we could redefine routes and see the result without restarting anything.

We made six decisions worth recording.

We let the server drive the animation rather than writing client-side JS. Each beat is an HTMX request that swaps in the next state. This is idiomatic FastHTML, and it also mirrors the algorithm: one round-trip per run of the target model.

Routes are stateless. The tick number travels in the URL, as in `step?tick=3`, so any request fully describes what to render. Nothing on the server can get out of sync, and the back button came for free.

The data is pre-scripted instead of coming from live models. Each iteration is a dict holding the committed prefix, the guesses, the number accepted, and the correction, with each prefix equal to the previous prefix plus its accepted guesses plus its correction. We also included the guesses that get thrown away after the first rejection. The paper's figure omits them, but they were generated, and showing them makes the wasted work visible.

The colors follow the paper. Committed tokens are black, guesses grey, and accepted, rejected, and corrected tokens are green, red, and blue. The rejected token gets a strikethrough so red and blue can sit side by side.

The two tracks are stacked, standard on top as the baseline, and swap together inside one container so they cannot drift apart. Two ticks make one target-model run. The standard track reveals one token per odd tick at the same pace for the whole animation. It does not speed up after the speculative track finishes, which keeps the comparison honest.

Stepping is manual. We started with a timer, then switched to Back and Next buttons that advance one tick per click, so a viewer can sit on the verification beat as long as they like. A step counter and phase labels ("M_q guessing", "M_p verifying") show where you are and which model is working.

The payoff comes at the end: speculative decoding finishes all 14 tokens in 3 steps, then sits idle while standard decoding plods through its remaining 11.
```

Main changes: cut the bold-label bullet format (the AI triad symmetry problem), removed the em-dashes, and turned each decision into plain sentences that say the thing once. Happy to adjust the length or tone if you want something shorter.

## Title and Abstract

<!-- page 0 -->

# Fast Inference from Transformers via Speculative Decoding

**Yaniv Leviathan** \(^{*1}\) **Matan Kalman** \(^{*1}\) **Yossi Matias** \(^{1}\)

### Abstract

Inference from large autoregressive models like Transformers is slow - decoding \(K\) tokens takes \(K\) serial runs of the model. In this work we introduce *speculative decoding* - an algorithm to sample from autoregressive models faster *without any changes to the outputs*, by computing several tokens in parallel. At the heart of our approach lie the observations that (1) hard language-modeling tasks often include easier subtasks that can be approximated well by more efficient models, and (2) using speculative execution and a novel sampling method, we can make exact decoding from the large models faster, by running them in parallel on the outputs of the approximation models, potentially generating several tokens concurrently, and without changing the distribution. Our method can accelerate existing off-the-shelf models without retraining or architecture changes. We demonstrate it on T5-XXL and show a **2X-3X** acceleration compared to the standard T5X implementation, with identical outputs.


## §1 Introduction

## 1. Introduction

Large autoregressive models, notably large Transformers (Vaswani et al., 2017), are much more capable than smaller models, as is evidenced countless times in recent years e.g., in the text or image domains, like GPT-3 (Brown et al., 2020), LaMDA (Thoppilan et al., 2022), Parti (Yu et al., 2022), and PaLM (Chowdhery et al., 2022). Unfortunately, a single decode step from these larger models is significantly slower than a step from their smaller counterparts, and making things worse, these steps are done serially - decoding \(K\) tokens takes \(K\) serial runs of the model.

What is an autoregressive model?  Are the decode steps slow due to the size of the model or due to the autoregressive architecture?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great questions to start with! Let me take the first part.

An **autoregressive model** generates a sequence one element at a time, where each new element is predicted based on all the elements that came before it. For language models, this means generating text token-by-token (roughly, word-by-word), where each token depends on the tokens already generated.

Formally, the model factors the probability of a whole sequence into a product of conditional probabilities:

$$p(x_1, x_2, \dots, x_T) = \prod_{t=1}^{T} p(x_t \mid x_{<t})$$

where $x_{<t}$ means "all the tokens before position $t$". The paper uses exactly this notation: $p(x_t|x_{<t})$ the distribution we get from the model for a prefix $x_{<t}$.

So to generate token $x_5$, you first need $x_1, x_2, x_3, x_4$ — you can't skip ahead.

Before I get to your second question (which is really the crux of the whole paper!), does this idea of "each token depends on the previous ones" make sense so far?

Yes, this makes sense

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Now for your second question — and it's a sharp one, because the answer is **both, but they're two separate effects**:

**Model size** determines how slow a *single* decode step is. A bigger model has more parameters to load and more arithmetic to do per token, so one step takes longer.

**The autoregressive architecture** determines how many steps must happen *one after another*. Because $x_t$ depends on $x_{<t}$, you can't compute token 5 until you've finished token 4. So decoding $K$ tokens means $K$ **serial** runs — they can't be parallelized in the naive way.

The paper puts these together: decoding $K$ tokens takes $K$ serial runs of the model, and a single step from a large model is significantly slower than from a small one.

Here's the key insight that motivates the whole paper: there's actually a *third* factor hiding here. The authors observe that a single decode step is often **not** limited by raw computation at all, but by something else in the hardware.

Any guess what that bottleneck might be — what a large model has to do at *every* step regardless of how much math is involved?

Given the importance of large autoregressive models and specifically large Transformers, several approaches were developed to make inference from them faster. Some approaches aim to reduce the inference cost for *all* inputs equally (e.g. Hinton et al., 2015; Jaszczur et al., 2021; Hubara et al., 2016; So et al., 2021; Shazeer, 2019). Other approaches stem from the observation that not all inference steps are born alike - some require a very large model, while others can be approximated well by more efficient models. These *adaptive computation* methods (e.g. Han et al., 2021; Sukhbaatar et al., 2019; Schuster et al., 2021; Scardapane et al., 2020; Bapna et al., 2020; Elbayad et al., 2019; Schwartz et al., 2020) aim to use less compute resources for easier inference steps. While many of these solutions have proven extremely effective in practice, they usually require changing the model architecture, changing the training-procedure and re-training the models, and don't maintain identical outputs.

The key observation above, that some inference steps are "harder" and some are "easier", is also a key motivator for our work. We additionally observe that inference from large models is often not bottlenecked on arithmetic operations, but rather on memory bandwidth and communication, so additional computation resources might be available. Therefore we suggest increasing concurrency as a complementary approach to using an adaptive amount of computation. Specifically, we are able to accelerate inference without changing the model architectures, without changing the training-procedures or needing to re-train the models, and without changing the model output distribution. This is accomplished via *speculative execution*.

Notes:
- inference from large models is bottlenecked on memory bandwidth and communication, not arithmetic operations
- increase concurrency, alongside an adaptive amount of computation
- contribution: accelerate inference **without**
    - changing the model architectures
    - changing training procedures
    - changing the model output distribution

Speculative execution (Burton, 1985; Hennessy & Patterson, 2012) is an optimization technique, common in processors, where a task is performed in parallel to verifying if it's actually needed - the payoff being increased concurrency. A well-known example of speculative execution is branch prediction. For speculative execution to be effective, we need an efficient mechanism to suggest tasks to execute that are likely to be needed. In this work, we generalize speculative execution to the stochastic setting - where a task *might* be needed with some probability. Applying this to decoding from autoregressive models like Transformers, we sample generations from more efficient *approximation models* as speculative prefixes for the slower *target models*. With a novel sampling method, *speculative sampling*, we maximize the probability of these speculative tasks to be accepted, while guaranteeing that the outputs from our system have the same distribution as those from the target model alone. For example, the sentence in Figure 1, consisting of 38 tokens, was generated by our method with only 9 serial runs of a larger target model (97M parameters) thanks to a smaller and more efficient approximation model (6M parameters), while the probability of generating it is unchanged.

We analyze our method in a variety of tasks and model sizes: unconditional generation from a 97M parameter GPT-like model trained on lm1b, English to German translation and news article summarization with an 11B parameters T5-XXL model, and a dialog task with a 137B parameter LaMDA model. We implement our method and compare actual walltimes for T5-XXL to those of the robust T5X implementation (Roberts et al., 2022), showing an out-of-the-box latency improvement of **2X-3X**, without any change to the outputs (Section 4).

Our method is easy to employ in actual production settings, doesn't require training new models, and doesn't change the outputs. Therefore, in common situations where memory bandwidth is the bottleneck, and compute resources are available, it may be a good default to accelerate sampling from autoregressive models like Transformers.

To summarize, our main contributions are: (1) A generalization of speculative execution to the stochastic setting, with a novel sampling method we call *speculative sampling*, and (2) A decoding mechanism we call *speculative decoding* that can accelerate decoding from autoregressive models, without any change to the model architectures, training regimes and output distributions.

![Figure 1](speculative-decoding//1_figure_1.png)

*Figure 1. Our technique illustrated in the case of unconditional language modeling. Each line represents one iteration of the algorithm. The **green** tokens are the suggestions made by the approximation model (here, a GPT-like Transformer decoder with 6M parameters trained on lm1b with 8k tokens) that the target model (here, a GPT-like Transformer decoder with 97M parameters in the same setting) accepted, while the **red** and **blue** tokens are the rejected suggestions and their corrections, respectively. For example, in the first line the target model was run only once, and 5 tokens were generated.*

## §2 Speculative Decoding — §2.1 Overview

## 2. Speculative Decoding

### 2.1. Overview

Let \( M_p \) be the target model, inference from which we're trying to accelerate, and \( p(x_t|x_{<t}) \) the distribution we get from the model for a prefix \( x_{<t} \). Let \( M_q \) be a more efficient approximation model for the same task, and denote by \( q(x_t|x_{<t}) \) the distribution we get from the model for a prefix \( x_{<t} \)<sup>1</sup>. The core idea is to (1) use the more efficient model \( M_q \) to generate \( \gamma \in \mathbb{Z}^+ \) completions (see Section 3.5 for how to optimally choose this parameter), then (2) use the target model \( M_p \) to evaluate all of the guesses and their respective probabilities from \( M_q \) *in parallel*, accepting all those that *can* lead to an identical distribution, and (3) sampling an additional token from an adjusted distribution to fix the first one that was rejected, or to add an additional one if they are all accepted. That way, each parallel run of the target model \( M_p \) will produce at least one new token (so the number of serial runs of the target model can never, even in the worst case, be larger than the simple autoregressive method), but it can potentially generate many new tokens, up to \( \gamma + 1 \), depending on how well \( M_q \) approximates \( M_p \).

## §2.2 Standardized Sampling

### 2.2. Standardized Sampling

First, note that while there are many methods and parameters of sampling, like argmax, top-k, nucleus, and setting a temperature, and popular implementations usually treat them differently at the logits level, they can all easily be cast into standard sampling from an adjusted probability distribution. For example, argmax sampling is equivalent to zeroing out non-max elements of the distribution and normalizing. We can therefore only deal with standard sampling from a probability distribution, and cast all of the other types of sampling into that framework. Going forward we’ll assume that \(p(x)\) and \(q(x)\) are the distributions from \( M_p \) and \( M_q \) respectively, adjusted for the sampling method.

---

<sup>1</sup>We'll use \( p(x) \) to denote \( p(x_t|x_{<t}) \) whenever the prefix \( x_{<t} \) is clear from the context, and similarly for \( q(x) \).

<!-- page 2 -->


## §2.3 Speculative Sampling

## 2.3. Speculative Sampling

To sample \( x \sim p(x) \), we instead sample \( x \sim q(x) \), keeping it if \( q(x) \leq p(x) \), and in case \( q(x) > p(x) \) we reject the sample with probability \( 1 - \frac{p(x)}{q(x)} \) and sample \( x \) again from an adjusted distribution \( p'(x) = norm(max(0, p(x) - q(x))) \) instead. It's easy to show (see Appendix A.1) that for any distributions \( p(x) \) and \( q(x) \), and \( x \) sampled in this way, indeed \( x \sim p(x) \).

Given the distribution \( q(x) \) obtained from running \( M_q \) on a conditioning *prefix*, we can sample a token \( x_1 \sim q(x) \). We then calculate the distribution \( p(x) \) by running \( M_p \) on *prefix* while in parallel speculatively calculating the distribution of the next token \( x_2 \) by running \( M_p \) on \( prefix + [x_1] \). Once both computations complete, we proceed as per above: If \( x_1 \) is rejected, we discard the computation of \( x_2 \) and re-sample \( x_1 \) from an adjusted distribution, and if \( x_1 \) is accepted, we keep both tokens. Algorithm 1 generalizes this idea to sample between 1 and \( \gamma + 1 \) tokens at once.

---
**Algorithm 1** SpeculativeDecodingStep
---
**Inputs:** \( M_p, M_q, prefix \).
▷ Sample \( \gamma \) guesses \( x_{1,...,\gamma} \) from \( M_q \) autoregressively.
**for** \( i = 1 \) **to** \( \gamma \) **do**
\( q_i(x) \leftarrow M_q(prefix + [x_1, \dots, x_{i-1}]) \)
\( x_i \sim q_i(x) \)
**end for**
▷ Run \( M_p \) in parallel.
\( p_1(x), \dots, p_{\gamma+1}(x) \leftarrow M_p(prefix), \dots, M_p(prefix + [x_1, \dots, x_\gamma]) \)
▷ Determine the number of accepted guesses \( n \).
\( r_1 \sim U(0, 1), \dots, r_\gamma \sim U(0, 1) \)
\( n \leftarrow \min(\{i - 1 \mid 1 \leq i \leq \gamma, r_i > \frac{p_i(x)}{q_i(x)}\} \cup \{\gamma\}) \)
▷ Adjust the distribution from \( M_p \) if needed.
\( p'(x) \leftarrow p_{n+1}(x) \)
**if** \( n < \gamma \) **then**
\( p'(x) \leftarrow norm(max(0, p_{n+1}(x) - q_{n+1}(x))) \)
**end if**
▷ Return one token from \( M_p \), and \( n \) tokens from \( M_q \).
\( t \sim p'(x) \)
**return** \( prefix + [x_1, \dots, x_n, t] \)
---


## §3 Analysis — §3.1 Number of Generated Tokens

## 3. Analysis

### 3.1. Number of Generated Tokens

Let's analyze the reduction factor in the number of serial calls to the target model, or equivalently, the expected number of tokens produced by a single run of Algorithm 1.

**Definition 3.1.** The *acceptance rate* \( \beta_{x_{<t}} \), given a prefix \( x_{<t} \), is the probability of accepting \( x_t \sim q(x_t|x_{<t}) \) by speculative sampling, as per Section 2.3\(^2\).

\( E(\beta) \) is then a natural measure of how well \( M_q \) approximates \( M_p \). If we make the simplifying assumption that the \( \beta \)s are i.i.d., and denote \( \alpha = E(\beta) \), then the number of tokens produced by a single run of Algorithm 1 is a capped geometric variable, with success probability \( 1 - \alpha \) and cap \( \gamma + 1 \), and the expected number of tokens generated by Algorithm 1 satisfies Equation (1). See Figure 2.

\[ E(\# \text{ generated tokens}) = \frac{1 - \alpha^{\gamma+1}}{1 - \alpha} \tag{1} \]

![Figure 2](speculative-decoding//2_figure_2.png)
*Figure 2. The expected number of tokens generated by Algorithm 1 as a function of \( \alpha \) for various values of \( \gamma \).*


## §3.2 Calculating α

### 3.2. Calculating \( \alpha \)

We'll now derive a simple formula for calculating \( \alpha \) given a prefix and the two models \( M_p \) and \( M_q \). We start by defining a natural divergence \( D_{LK} \):

**Definition 3.2.** \( D_{LK}(p, q) = \sum_x |p(x) - M(x)| = \sum_x |q(x) - M(x)| \) where \( M(x) = \frac{p(x)+q(x)}{2} \).

**Lemma 3.3.** \( D_{LK}(p, q) = 1 - \sum_x \min(p(x), q(x)) \)

**Proof.** \( D_{LK}(p, q) = \sum_x |p(x) - M(x)| = \sum_x \frac{|p-q|}{2} = 1 - \sum_x \frac{p+q-|p-q|}{2} = 1 - \sum_x \min(p(x), q(x)) \) ◻

From Lemma 3.3 we immediately get the following results:

---
\(^2\)As before, we'll omit the \( x_{<t} \) subscript whenever the prefix is clear from the context.

<!-- page 3 -->

### Fast Inference from Transformers via Speculative Decoding

**Corollary 3.4.** \( D_{LK}(p, q) \) *is a symmetric divergence in \( [0, 1] \).*
\( D_{LK}(p, q) = 0 \iff p = q \).
\( D_{LK}(p, q) = 1 \iff p \text{ and } q \text{ have disjoint support} \).

**Theorem 3.5.** \( \beta = 1 - D_{LK}(p, q) \)

*Proof.* \[
\beta = E_{x \sim q(x)} \begin{cases} 1 & q(x) \le p(x) \\ \frac{p(x)}{q(x)} & q(x) > p(x) \end{cases} = E_{x \sim q(x)} \min(1, \frac{p(x)}{q(x)}) = \sum_x \min(p(x), q(x))
\]

Finally we get:
**Corollary 3.6.** \( \alpha = 1 - E(D_{LK}(p, q)) = E(\min(p, q)) \)

See Table 3 for empirically observed \( \alpha \) values in our experiments.


## §3.3 Walltime Improvement

### 3.3. Walltime Improvement

We’ve shown that with the i.i.d. assumption our algorithm reduces the number of calls to the target model by a factor of \( \frac{1-\alpha^{\gamma+1}}{1-\alpha} \). Note that speculative execution in general, and our algorithm in particular, assume that we have enough compute resources to support the increased concurrency (Section 3.4). For the walltime analysis, we’ll assume that we can run \( \gamma + 1 \) concurrent evaluations of \( M_p \) in parallel without increasing the walltime. To get the total walltime improvement, we now consider the cost of running the approximation model \( M_q \).

**Definition 3.7.** Let \( c \), the *cost coefficient*, be the ratio between the time for a single run of \( M_q \) and the time for a single run of \( M_p \).

Note that unlike \( \alpha \) which is an intrinsic property of the models and the task, the value of \( c \) depends on the hardware configuration and software implementation details. In our experiments where \( M_q \) is typically a couple of orders of magnitude smaller than \( M_p \), \( c \) was always less than 0.05 and often negligibly close to 0.

**Theorem 3.8.** *The expected improvement factor in total walltime by Algorithm 1 is \( \frac{1-\alpha^{\gamma+1}}{(1-\alpha)(\gamma c+1)} \).*

*Proof.* Denote the cost of running a single step of \( M_p \) by \( T \). Now, each run of Algorithm 1 costs \( T c \gamma + T \) (for running the approximation model \( M_q \) \( \gamma \) times and running \( M_p \) once) and according to Equation (1) produces \( \frac{1-\alpha^{\gamma+1}}{1-\alpha} \) tokens on average. So the overall expected cost for producing a token with Algorithm 1 is \( \frac{(c \gamma+1)(1-\alpha)}{1-\alpha^{\gamma+1}} T \). Since the cost of producing a single token with the standard decoding algorithm is \( T \), we get the desired result. ◻

Note that Theorem 3.8 assumes long enough generations (for example, since we run \( M_p \) at least once, the improvement factor is capped by the number of generated tokens).

**Corollary 3.9.** *If \( \alpha > c \), there exists \( \gamma \) for which we’ll get an improvement, and the improvement factor will be at least \( \frac{1+\alpha}{1+c} \).*

*Proof.* If we get an improvement for \( \gamma \), we’d also get an improvement for any \( 0 < \gamma^* < \gamma \), so for our method to yield an improvement, we can evaluate Theorem 3.8 for \( \gamma = 1 \), yielding \( \frac{1-\alpha^2}{(1-\alpha)(c+1)} = \frac{1+\alpha}{1+c} \). ◻


## §3.4 Number of Arithmetic Operations

### 3.4. Number of Arithmetic Operations

Algorithm 1 does \( \gamma+1 \) runs of \( M_p \) in parallel, so the number of *concurrent* arithmetic operations grows by a factor of \( \gamma+1 \). Now, since Algorithm 1 produces at most \( \gamma+1 \) tokens per run, the *total* number of arithmetic operations might be higher than that of the standard decoding algorithm. When we accept the sample from \( M_q \) the increased concurrency is ‘‘free’’ and the total number of operations isn’t increased⁳. When we reject a guess though, computation is wasted. Let’s now analyze the effect of our method on the total number of arithmetic operations.

**Definition 3.10.** Let \( \hat{c} \) be the ratio of arithmetic operations per token of the approximation model \( M_q \) to that of the target model \( M_p \).

**Theorem 3.11.** *The expected factor of increase in the number of total operations of Algorithm 1 is \( \frac{(1-\alpha)(\gamma\hat{c}+\gamma+1)}{1-\alpha^{\gamma+1}} \).*

*Proof.* Denote by \( \hat{T} \) the number of arithmetic operations done by a standard decoding baseline per token, i.e. the number of operations of a single run of \( M_p \). Then a single iteration of Algorithm 1 costs \( \hat{T}\hat{c}\gamma + \hat{T}(\gamma + 1) \) operations (for \( \gamma \) runs of \( M_q \) and \( \gamma + 1 \) parallel runs of \( M_p \)). Dividing by the expected number of tokens produced by Algorithm 1, i.e. Equation (1), and by \( \hat{T} \), we get the desired result. ◻

If \( \alpha \) is low, the increase in the number of arithmetic operations is high, and vice-versa. Note that for Transformer decoders, the total number of arithmetic operations by Algorithm 1 (not counting runs of \( M_q \)) can be bounded from above by a single run of the same-size Transformer encoder.

Unlike the total number of arithmetic operations, the total number of memory accesses can go down with our method. Specifically, the target model’s weights and KV cache can be read once per execution of Algorithm 1, so the number of memory accesses for reading them shrinks by a factor of \( \frac{1-\alpha^{\gamma+1}}{1-\alpha} \), according to Equation (1).


## §3.5 Choosing γ

### 3.5. Choosing \( \gamma \)

Given \( c \) and \( \alpha \) and assuming enough compute resources (see Section 3.4), the optimal \( \gamma \) is the one maximizing the wall-

———————
⁳ Neglecting the cost of \( M_q \).

<!-- page 4 -->

## Fast Inference from Transformers via Speculative Decoding

![Figure 3](speculative-decoding//4_figure_3.png)

*Figure 3. The optimal \( \gamma \) as a function of \( \alpha \) for various values of \( c \).*

time improvement equation (Theorem 3.8): \( \frac{1-\alpha^{\gamma+1}}{(1-\alpha)(\gamma c+1)} \). Since \( \gamma \) is an integer, it can be easily found numerically, see Figure 3.

Table 1 and Figure 4 illustrate the trade-off between inference speed and the total number of arithmetic operations for various values of \( \alpha \) and \( \gamma \), assuming \( c = \hat{c} = 0 \). Figure 5 shows a simplified trace diagram.

*Table 1. The total number of arithmetic operations and the inference speed vs the baseline, for various values of \( \gamma \) and \( \alpha \), assuming \( c = \hat{c} = 0 \).*

| \( \alpha \) | \( \gamma \) | OPERATIONS | SPEED |
| :--- | :--- | :--- | :--- |
| 0.6 | 2 | 1.53X | 1.96X |
| 0.7 | 3 | 1.58X | 2.53X |
| 0.8 | 2 | 1.23X | 2.44X |
| 0.8 | 5 | 1.63X | 3.69X |
| 0.9 | 2 | 1.11X | 2.71X |
| 0.9 | 10 | 1.60X | 6.86X |

Instead of picking a single value for \( \gamma \) based on \( \alpha \), since the \( \beta \)s aren't constant, we could get further improvement by predicting the value of \( \beta \) and accordingly varying the value of \( \gamma \) during the run of Algorithm 1. To get an upper bound on the additional improvement factor, assume we had an oracle for \( \gamma \). We would then have \( E(\# \text{ generated tokens}) = \frac{1}{1-\alpha} \). For typical values of \( c \) and \( \alpha \), and assuming unbounded compute resources, the enhanced walltime improvement factor can be up to \( \sim 60\% \) higher than the improvement factor with a fixed \( \gamma \). We leave exploring this for future work<sup>4</sup>.


## §3.6 Approximation Models

### 3.6. Approximation Models

Speculative sampling, and therefore speculative decoding, guarantee an identical output distribution for any choice of approximation model \( M_q \) without restriction (see Appendix A.1). In our experiments, we mostly tested existing off-the-shelf smaller Transformers as the approximation models. Further, we only tested approximation models of the same architecture as the target models \( M_p \) and using the same probability standardization. In this setup, choosing \( M_q \) to be around two orders of magnitude smaller than \( M_p \) usually performed best, balancing \( \alpha \) and \( c \) (Theorem 3.8).

Another type of approximation models, *negligible-cost models*, are those for which \( c \approx 0 \), i.e. approximation models with a negligible cost relative to the target model. In this case, we get an expected walltime improvement of \( \frac{1-\alpha^{\gamma+1}}{1-\alpha} \), which is bounded from above by \( \frac{1}{1-\alpha} \) (we approach equality if \( \gamma \) is large). One interesting type of negligible-cost approximation models are n-gram models, where the evaluation amounts to a table lookup. Interestingly, in empirical tests (Section 4.2) we get non zero \( \alpha \)s even for these trivial n-gram models. For example, for the English-German translation task, with \( M_p \) being T5-XXL 11B and \( M_q \) being a trivial bigram model, we get \( \alpha \approx 0.2 \) which leads to an inference speed improvement factor of 1.25X with \( \gamma = 3 \).

Other simple heuristics can be used as negligible-cost approximation models. For example, in cases where long sequences are likely to repeat, such as for summarization tasks or chat-like interfaces<sup>5</sup>, an approximation model that simply

![Figure 4](speculative-decoding//4_figure_4.png)

*Figure 4. The speedup factor and the increase in number of arithmetic operations as a function of \( \alpha \) for various values of \( \gamma \).*

---
<sup>4</sup>The above bound assumes that we still run \( M_p \) to verify the oracle's predictions. If we skip those verifications the bound doesn't hold and we would get a substantial additional improvement.

<sup>5</sup>E.g. where a user and a language model iterate on content, like text or code ("can you rewrite this story but change the ending", "can you make this function also do X").

<!-- page 5 -->

![Figure 5](speculative-decoding//5_figure_5.png)

*Figure 5. A simplified trace diagram for a full encoder-decoder Transformer stack. The top row shows speculative decoding with \(\gamma = 7\) so each of the calls to \(M_p\) (the purple blocks) is preceded by 7 calls to \(M_q\) (the blue blocks). The yellow block on the left is the call to the encoder for \(M_p\) and the orange block is the call to the encoder for \(M_q\). Likewise the middle row shows speculative decoding with \(\gamma = 3\), and the bottom row shows standard decoding.*

copies tokens from the context in case we find a matching prefix, might yield high values of \(\alpha\). These parameter-less approximation models, have the additional advantage of being even simpler to deploy from a production standpoint.

Another type of approximation models that can be used by speculative decoding are non-autoregressive models, like those from (Stern et al., 2018). Then, instead of the autoregressive loop in Algorithm 1 we'd just call the non-autoregressive model once.

A final example, interesting mostly from a theoretical perspective, is an approximation model which chooses tokens at random, which guarantees some improvement (although very small) for all models \(M_p\).


## §4 Experiments

### 4. Experiments

#### 4.1. Empirical Walltime Improvement

We implement our algorithm and compare it to the implementation in the T5X codebase for accelerating T5-XXL.

**Setup** We test a standard encoder-decoder T5 version 1.1 model (Raffel et al., 2020) on two tasks from the T5 paper: (1) English to German translation fine tuned on WMT EnDe, and (2) Text summarization fine tuned on CCN/DM. For both tasks, we use T5-XXL (11B) for \(M_p\). For the approximation model \(M_q\) we test several existing configurations, namely T5-large (800M), T5-base (250M), and T5-small (77M) (Raffel et al., 2020). We use existing checkpoints for all models. We measure walltime improvements with a batch size of 1 on a single TPU-v4 for both argmax sampling (temp=0) and standard sampling (temp=1).

**Results** Table 2 shows the empirical results from our method. We see that T5-small (77M), with a good balance of \(c\) and \(\alpha\), provides the highest speedup out of the tested approximation models. As expected we see that \(\alpha\) increases with the size of the approximation model. Interestingly, \(\alpha\) and walltime improvement are higher for argmax sampling (temp=0). We observe speedups of 2.6X (temp=1) and 3.4X (temp=0) on the translation task and slightly lower speedups of 2.3X (temp=1) and 3.1X (temp=0) for the summarization task. These empirical results match well with the theoretical predictions, with some variance due to implementation details (see Appendix A.3).

*Table 2. Empirical results for speeding up inference from a T5-XXL 11B model.*

| TASK | \(M_q\) | TEMP | \(\gamma\) | \(\alpha\) | SPEED |
| :--- | :--- | :--- | :--- | :--- | :--- |
| ENDE | T5-SMALL ★ | 0 | 7 | 0.75 | **3.4X** |
| ENDE | T5-BASE | 0 | 7 | 0.8 | 2.8X |
| ENDE | T5-LARGE | 0 | 7 | 0.82 | 1.7X |
| ENDE | T5-SMALL ★ | 1 | 7 | 0.62 | **2.6X** |
| ENDE | T5-BASE | 1 | 5 | 0.68 | 2.4X |
| ENDE | T5-LARGE | 1 | 3 | 0.71 | 1.4X |
| CNNDM | T5-SMALL ★ | 0 | 5 | 0.65 | **3.1X** |
| CNNDM | T5-BASE | 0 | 5 | 0.73 | 3.0X |
| CNNDM | T5-LARGE | 0 | 3 | 0.74 | 2.2X |
| CNNDM | T5-SMALL ★ | 1 | 5 | 0.53 | **2.3X** |
| CNNDM | T5-BASE | 1 | 3 | 0.55 | 2.2X |
| CNNDM | T5-LARGE | 1 | 3 | 0.56 | 1.7X |

#### 4.2. Empirical \(\alpha\) Values

While we only implemented our method for T5, we measured \(\alpha\) values for various tasks, sampling methods, target models \(M_p\), and approximation models \(M_q\). Specifically, we evaluated the expectation from Corollary 3.6 on 10K tokens generated by \(M_p\), for each of the settings below.

**GPT-like (97M params)** We test a decoder-only Transformer model on unconditional language generation, trained on lm1b (Chelba et al., 2013). The model here is a GPT-like Transformer decoder with Gelu activations (Hendrycks & Gimpel, 2016). For \(M_q\) we experimented with a Trans

<!-- page 6 -->

former decoder model with 6M parameters: dim 256, dim feed-forward 1024, 2 layers, 4 attention heads, as well as simple unigram and bigram models. \(M_p\) has 97M parameters: dim 768, dim feed-forward 3072, 12 layers, 12 attention heads. We used Bert tokenization (Devlin et al., 2019) with 8k tokens for all models.

**LaMDA (137B params)** We tested a decoder only LaMDA model on a dialog task (Thoppilan et al., 2022). We used existing checkpoints from LaMDA 137B as \(M_p\) and LaMDA 8B, LaMDA 2B, and LaMDA 100M for \(M_q\).

See Section 4.1 for the setup of the T5-XXL (11B params) model.

Table 3 summarizes the \(\alpha\) values for the tested cases. We observe that approximation models that are a couple of orders of magnitude smaller than the target model tend to produce \(\alpha\) values between 0.5 and 0.9. Interestingly, we also note that for all models, the sharper the adjusted distribution, the higher the \(\alpha\) values. Finally, we note that even trivial unigram and bigram approximations yield non negligible \(\alpha\) values. For example, for the case of English to German translation, the bigram model has an \(\alpha\) value of 0.2, and since \(c = 0\) in this case, yields a 1.25X speed improvement, which is surprisingly high for this trivial approximation model (but is still lower than the speedup we get from using T5-small as the approximation model).


## §5 Related Work

### 5. Related work

The efficiency of inference from large models was studied extensively (Dehghani et al., 2021). Many approaches aim to speed up inference from large models in general, and autoregressive models like Transformers in particular. Numerous techniques try to make inference more efficient for all tokens, e.g. distillation (Hinton et al., 2015), sparcification (Jaszczur et al., 2021), quantization (Hubara et al., 2016), and architecture modification (So et al., 2021; Shazeer, 2019). Closer to our approach are adaptive computation methods which adapt the amount of computation to problem difficulty (Han et al., 2021). Examples include attending to a subset of the inputs (Sukhbaatar et al., 2019), and early exits (Schuster et al., 2021; Scardapane et al., 2020; Bapna et al., 2020; Elbayad et al., 2019; Schwartz et al., 2020). Notably, Wisdom of Committees (Schwartz et al., 2020) leverages off-the-shelf smaller models, but is an adaptive computation approach, and so it uses a heuristic to determine when to stop, losing the guarantee of identical outputs to those of the target models. In general, adaptive computation methods usually learn, either within the model itself or with an auxiliary model, when a computation shortcut can be taken. Usually, these methods save on both inference time and arithmetic operations, but require a change of architecture, a change of training procedure and training custom models or

| \(M_p\) | \(M_q\) | SMPL | \(\alpha\) |
| :--- | :--- | :--- | :--- |
| GPT-LIKE (97M) | UNIGRAM | T=0 | 0.03 |
| GPT-LIKE (97M) | BIGRAM | T=0 | 0.05 |
| GPT-LIKE (97M) | GPT-LIKE (6M) | T=0 | 0.88 |
| GPT-LIKE (97M) | UNIGRAM | T=1 | 0.03 |
| GPT-LIKE (97M) | BIGRAM | T=1 | 0.05 |
| GPT-LIKE (97M) | GPT-LIKE (6M) | T=1 | 0.89 |
| T5-XXL (ENDE) | UNIGRAM | T=0 | 0.08 |
| T5-XXL (ENDE) | BIGRAM | T=0 | 0.20 |
| T5-XXL (ENDE) | T5-SMALL | T=0 | 0.75 |
| T5-XXL (ENDE) | T5-BASE | T=0 | 0.80 |
| T5-XXL (ENDE) | T5-LARGE | T=0 | 0.82 |
| T5-XXL (ENDE) | UNIGRAM | T=1 | 0.07 |
| T5-XXL (ENDE) | BIGRAM | T=1 | 0.19 |
| T5-XXL (ENDE) | T5-SMALL | T=1 | 0.62 |
| T5-XXL (ENDE) | T5-BASE | T=1 | 0.68 |
| T5-XXL (ENDE) | T5-LARGE | T=1 | 0.71 |
| T5-XXL (CNNDM) | UNIGRAM | T=0 | 0.13 |
| T5-XXL (CNNDM) | BIGRAM | T=0 | 0.23 |
| T5-XXL (CNNDM) | T5-SMALL | T=0 | 0.65 |
| T5-XXL (CNNDM) | T5-BASE | T=0 | 0.73 |
| T5-XXL (CNNDM) | T5-LARGE | T=0 | 0.74 |
| T5-XXL (CNNDM) | UNIGRAM | T=1 | 0.08 |
| T5-XXL (CNNDM) | BIGRAM | T=1 | 0.16 |
| T5-XXL (CNNDM) | T5-SMALL | T=1 | 0.53 |
| T5-XXL (CNNDM) | T5-BASE | T=1 | 0.55 |
| T5-XXL (CNNDM) | T5-LARGE | T=1 | 0.56 |
| LAMDA (137B) | LAMDA (100M) | T=0 | 0.61 |
| LAMDA (137B) | LAMDA (2B) | T=0 | 0.71 |
| LAMDA (137B) | LAMDA (8B) | T=0 | 0.75 |
| LAMDA (137B) | LAMDA (100M) | T=1 | 0.57 |
| LAMDA (137B) | LAMDA (2B) | T=1 | 0.71 |
| LAMDA (137B) | LAMDA (8B) | T=1 | 0.74 |

*Table 3. Empirical \(\alpha\) values for various target models \(M_p\), approximation models \(M_q\), and sampling settings. T=0 and T=1 denote argmax and standard sampling respectively.* 6

re-training of existing models. They usually also change the outputs of the model. We note that while many of the methods above improve the memory to arithmetic-operations ratio, in cases where the ratio remains high, these methods and our speculative decoding method might be effective in tandem.

Two prior methods leverage speculative execution for speeding up decoding from autoregressive models. Blockwise Parallel Decoding (Stern et al., 2018) decodes several tokens in parallel, similarly to our work. However, it only supports greedy decoding (temperature=0) and not the general stochastic setting, it requires additional training of a custom model, and focuses on preserving down-stream task quality, instead of guaranteeing identical outputs. Shallow Aggressive Decoding (SAD) (Sun et al., 2021) also decodes several tokens in parallel, similarly to our work. Unlike our work, SAD only supports copying the input to the output.

<!-- page 7 -->

put, and not general approximation models, making it only
suitable for the cases where the inputs and outputs are very
similar like grammatical error correction. In addition, simi-
larly to Blockwise Parallel Decoding, SAD does not support
the general stochastic sampling setting.

After we initially published our work, an independent im-
plementation of speculative decoding (Chen et al., 2023)
showed similar 2X-2.5X improvements on Chinchilla 70B.


## §6 Discussion

### 6. Discussion

We presented *speculative sampling* which enables efficient
*stochastic speculative execution* - i.e. speculative execu-
tion in the stochastic setting. We analyzed its impact on
decoding from autoregressive models like Transformers via
*speculative decoding* and have shown that given enough
compute resources, we get meaningful 2X-3X speedups in
practice vs T5X, a popular optimized implementation.

One limitation of speculative execution in general, and of
speculative decoding in particular, is that latency is im-
proved through increased concurrency at the cost of an in-
creased number of arithmetic operations. Thus, our method
is not helpful for configurations where additional compu-
tation resources are not available. However, in common
cases where additional computation resources are available
(e.g. when memory bandwidth is the bottleneck) our method
provides the speedup with significant benefits: the model
architecture doesn’t change, retraining isn’t required, and
most importantly, *the output distribution is guaranteed to
stay the same*. Our method is easy to implement, and can
be used to speedup inference using out-of-the-box models
without developing and evaluating custom schemes.

There are several directions for follow up research, impor-
tantly, further investigating the compatibility of speculative
decoding with beam search (see Appendix A.4). Also, while
our method yields substantial speedups with existing off-the-
shelf approximation models, greater improvements might
be obtained via custom approximation models (Section 3.6),
such as those with custom architectures (e.g. custom sizes,
non-autoregressive models, or various heuristics) or with
custom training procedures (e.g. standard distillation with
soft targets from \( M_p \), or optimizing \( M_q \) for \( \alpha \) directly). It
could also be interesting to explore a hierarchical version
of the algorithm, where the approximation model is itself
accelerated by an even faster model, which could allow
for more capable approximation models. In this work we
fixed the approximation model and the number of guesses
\( \gamma \) throughout inference, but varying them during inference
could yield additional improvements (Section 3.5). In our

\rule{150pt}{0.5pt}

\(^6\)Note that the outputs from the LaMDA model always go
through a \( Top_{40} \) filter. This has no effect on argmax, but does
have some effect on standard sampling.

experiments we always performed the same standardization
on the distributions generated by the approximation model
as the desired one for the target model (Section 2.2), but fur-
ther improvements might be obtained by applying different
transformations. We tested speculative decoding only in the
text modality, but it might work well in other domains (e.g.
images) which would be interesting to experiment with.

Finally, we note that *stochastic speculative execution* and
*speculative sampling* can be helpful outside the scope of
*speculative decoding* from autoregressive models. For ex-
ample, given two slow functions, \( f(x) \) and \( g(y) \) such that
\( f(x) \) generates a distribution from which \( g \)'s input is sam-
pled, we could use our method to run \( f \) and \( g \) in parallel.
This setup might arise e.g. in physics simulations, or in rein-
forcement learning where \( f \) is a large model that produces a
distribution on actions, and \( g \) is the world simulation, which
would be interesting to explore.


## Acknowledgments & References

### Acknowledgments

We would like to extend a special thank you to YaGuang Li
for help with everything LaMDA related and for calculating
the LaMDA figures in the paper, and to Blake Hechtman
for great insights and help with XLA. We would also like
to thank the reviewers for insightful comments, as well
as Asaf Aharoni, Reiner Pope, Sasha Goldshtein, Nadav
Sherman, Eyal Segalis, Eyal Molad, Dani Valevski, Daniel
Wasserman, Valerie Nygaard, Danny Vainstein, the LaMDA
and Theta Labs teams at Google, and our families.

### References

Bapna, A., Arivazhagan, N., and Firat, O. Controlling
computation versus quality for neural sequence models.
*ArXiv*, abs/2002.07106, 2020.

Brown, T. B., Mann, B., Ryder, N., Subbiah, M., Kaplan,
J., Dhariwal, P., Neelakantan, A., Shyam, P., Sastry, G.,
Askell, A., Agarwal, S., Herbert-Voss, A., Krueger, G.,
Henighan, T., Child, R., Ramesh, A., Ziegler, D. M., Wu,
J., Winter, C., Hesse, C., Chen, M., Sigler, E., Litwin, M.,
Gray, S., Chess, B., Clark, J., Berner, C., McCandlish,
S., Radford, A., Sutskever, I., and Amodei, D. Lan-
guage models are few-shot learners. In *Proceedings of
the 34th International Conference on Neural Informa-
tion Processing Systems*, NIPS'20, Red Hook, NY, USA,
2020. Curran Associates Inc. ISBN 9781713829546.

Burton, F. W. Speculative computation, parallelism, and
functional programming. *IEEE Transactions on Comput-
ers*, C-34(12):1190–1193, 1985. doi: 10.1109/TC.1985.
6312218.

Chelba, C., Mikolov, T., Schuster, M., Ge, Q., Brants, T.,
Koehn, P. T., and Robinson, T. One billion word bench-

<!-- page 8 -->

mark for measuring progress in statistical language modeling. In *Interspeech*, 2013.

Chen, C., Borgeaud, S., Irving, G., Lespiau, J.-B., Sifre, L., and Jumper, J. M. Accelerating large language model decoding with speculative sampling. *ArXiv, abs/2302.01318*, 2023.

Chowdhery, A., Narang, S., Devlin, J., Bosma, M., Mishra, G., Roberts, A., Barham, P., Chung, H. W., Sutton, C., Gehrmann, S., Schuh, P., Shi, K., Tsvyashchenko, S., Maynez, J., Rao, A., Barnes, P., Tay, Y., Shazeer, N. M., Prabhakaran, V., Reif, E., Du, N., Hutchinson, B. C., Pope, R., Bradbury, J., Austin, J., Isard, M., Gur-Ari, G., Yin, P., Duke, T., Levskaya, A., Ghemawat, S., Dev, S., Michalewski, H., García, X., Misra, V., Robinson, K., Fedus, L., Zhou, D., Ippolito, D., Luan, D., Lim, H., Zoph, B., Spiridonov, A., Sepassi, R., Dohan, D., Agrawal, S., Omernick, M., Dai, A. M., Pillai, T. S., Pellat, M., Lewkowycz, A., Moreira, E., Child, R., Polozov, O., Lee, K., Zhou, Z., Wang, X., Saeta, B., Díaz, M., Firat, O., Catasta, M., Wei, J., Meier-Hellstern, K. S., Eck, D., Dean, J., Petrov, S., and Fiedel, N. Palm: Scaling language modeling with pathways. *ArXiv, abs/2204.02311*, 2022.

Dehghani, M., Arnab, A., Beyer, L., Vaswani, A., and Tay, Y. The efficiency misnomer. *ArXiv, abs/2110.12894*, 2021.

Devlin, J., Chang, M.-W., Lee, K., and Toutanova, K. Bert: Pre-training of deep bidirectional transformers for language understanding. *ArXiv, abs/1810.04805*, 2019.

Elbayad, M., Gu, J., Grave, E., and Auli, M. Depth-adaptive transformer. *ArXiv, abs/1910.10073*, 2019.

Han, Y., Huang, G., Song, S., Yang, L., Wang, H., and Wang, Y. Dynamic neural networks: A survey. *IEEE Transactions on Pattern Analysis and Machine Intelligence*, 44: 7436–7456, 2021.

Hendrycks, D. and Gimpel, K. Bridging nonlinearities and stochastic regularizers with gaussian error linear units. *ArXiv, abs/1606.08415*, 2016.

Hennessy, J. L. and Patterson, D. A. *Computer Architecture: A Quantitative Approach*. Morgan Kaufmann, Amsterdam, 5 edition, 2012. ISBN 978-0-12-383872-8.

Hinton, G. E., Vinyals, O., and Dean, J. Distilling the knowledge in a neural network. *ArXiv, abs/1503.02531*, 2015.

Hubara, I., Courbariaux, M., Soudry, D., El-Yaniv, R., and Bengio, Y. Quantized neural networks: Training neural networks with low precision weights and activations. *ArXiv, abs/1609.07061*, 2016.

Jaszczur, S., Chowdhery, A., Mohiuddin, A., Kaiser, L., Gajewski, W., Michalewski, H., and Kanerva, J. Sparse is enough in scaling transformers. In *Neural Information Processing Systems*, 2021.

Raffel, C., Shazeer, N., Roberts, A., Lee, K., Narang, S., Matena, M., Zhou, Y., Li, W., and Liu, P. J. Exploring the limits of transfer learning with a unified text-to-text transformer. *The Journal of Machine Learning Research*, 21(1):5485–5551, 2020.

Roberts, A., Chung, H. W., Levskaya, A., Mishra, G., Bradbury, J., Andor, D., Narang, S., Lester, B., Gaffney, C., Mohiuddin, A., Hawthorne, C., Lewkowycz, A., Salcianu, A., van Zee, M., Austin, J., Goodman, S., Soares, L. B., Hu, H., Tsvyashchenko, S., Chowdhery, A., Bastings, J., Bulian, J., García, X., Ni, J., Chen, A., Kenealy, K., Clark, J., Lee, S., Garrette, D. H., Lee-Thorp, J., Raffel, C., Shazeer, N. M., Ritter, M., Bosma, M., Passos, A., Maitin-Shepard, J. B., Fiedel, N., Omernick, M., Saeta, B., Sepassi, R., Spiridonov, A., Newlan, J., and Gesmundo, A. Scaling up models and data with t5x and seqio. *ArXiv, abs/2203.17189*, 2022.

Scardapane, S., Scarpiniti, M., Baccarelli, E., and Uncini, A. Why should we add early exits to neural networks? *Cognitive Computation*, 12(5):954–966, 2020.

Schuster, T., Fisch, A., Jaakkola, T., and Barzilay, R. Consistent accelerated inference via confident adaptive transformers. In *Conference on Empirical Methods in Natural Language Processing*, 2021.

Schwartz, R., Stanovsky, G., Swayamdipta, S., Dodge, J., and Smith, N. A. The right tool for the job: Matching model and instance complexities. In *Annual Meeting of the Association for Computational Linguistics*, 2020.

Shazeer, N. M. Fast transformer decoding: One write-head is all you need. *ArXiv, abs/1911.02150*, 2019.

So, D. R., Ma’nke, W., Liu, H., Dai, Z., Shazeer, N. M., and Le, Q. V. Primer: Searching for efficient transformers for language modeling. *ArXiv, abs/2109.08668*, 2021.

Stern, M., Shazeer, N., and Uszkoreit, J. Blockwise parallel decoding for deep autoregressive models. *Advances in Neural Information Processing Systems*, 31, 2018.

Sukhbaatar, S., Grave, E., Bojanowski, P., and Joulin, A. Adaptive attention span in transformers. In *Annual Meeting of the Association for Computational Linguistics*, 2019.

Sun, X., Ge, T., Wei, F., and Wang, H. Instantaneous grammatical error correction with shallow aggressive decoding. *ArXiv, abs/2106.04970*, 2021.

9

<!-- page 9 -->

Thoppilan, R., Freitas, D. D., Hall, J., Shazeer, N. M., Kul-shreshtha, A., Cheng, H.-T., Jin, A., Bos, T., Baker, L., Du, Y., Li, Y., Lee, H., Zheng, H., Ghafouri, A., Menegali, M., Huang, Y., Krikun, M., Lepikhin, D., Qin, J., Chen, D., Xu, Y., Chen, Z., Roberts, A., Bosma, M., Zhou, Y., Chang, C.-C., Krivokon, I. A., Rusch, W. J., Pickett, M., Meier-Hellstern, K. S., Morris, M. R., Doshi, T., Santos, R. D., Duke, T., Søraker, J. H., Zevenbergen, B., Prabhakaran, V., Díaz, M., Hutchinson, B., Olson, K., Molina, A., Hoffman-John, E., Lee, J., Aroyo, L., Rajakumar, R., Butryna, A., Lamm, M., Kuzmina, V. O., Fenton, J., Cohen, A., Bernstein, R., Kurzweil, R., Aguera-Arcas, B., Cui, C., Croak, M., hsin Chi, E. H., and Le, Q. Lamda: Language models for dialog applications. *ArXiv*, abs/2201.08239, 2022.

Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, Ł., and Polosukhin, I. Attention is all you need. *Advances in neural information processing systems*, 30, 2017.

Yu, J., Xu, Y., Koh, J. Y., Luong, T., Baid, G., Wang, Z., Vasudevan, V., Ku, A., Yang, Y., Ayan, B. K., Hutchinson, B. C., Han, W., Parekh, Z., Li, X., Zhang, H., Baldridge, J., and Wu, Y. Scaling autoregressive models for content-rich text-to-image generation. *ArXiv*, abs/2206.10789, 2022.

<!-- page 10 -->


## Appendix A.1–A.3

## A. Appendix

### A.1. Correctness of Speculative Sampling

We will now show that for any distributions \( p(x) \) and \( q(x) \), the tokens sampled via *speculative sampling* from \( p(x) \) and \( q(x) \) are distributed identically to those sampled from \( p(x) \) alone. Let \( \beta \) be the acceptance probability (Definition 3.1).

Note that as \( p'(x) = norm(max(0, p(x) - q(x))) = \frac{p(x) - \min(q(x), p(x))}{\sum_{x'} (p(x') - \min(q(x'), p(x')))} = \frac{p(x) - \min(q(x), p(x))}{1 - \beta} \), the normalizing constant for the adjusted distribution \( p'(x) \) is \( 1 - \beta \), where the last equation follows immediately from Lemma 3.3 and Theorem 3.5.

Now:

\[ P(x = x') = P(guess\ accepted, x = x') + P(guess\ rejected, x = x') \]

Where:

\[ P(guess\ accepted, x = x') = q(x') \min(1, \frac{p(x')}{q(x')}) = \min(q(x'), p(x')) \]

And:

\[ P(guess\ rejected, x = x') = (1 - \beta)p'(x') = p(x') - \min(q(x'), p(x')) \]

Overall:

\[ P(x = x') = \min(p(x'), q(x')) + p(x') - \min(p(x'), q(x')) = p(x'). \]

As desired. \(\square\)

### A.2. Speculative Sampling vs. Rejection Sampling

Rejection sampling is the following iterative sampling procedure that looks superficially similar to ours:

1. Sample \( x \sim q(x) \) and \( r \sim U(0, 1) \).
2. If \( r < \frac{p(x)}{M q(x)} \) return \( x \).
3. Go to 1.

Where \( M = \max_x \frac{p(x)}{q(x)} \). We could employ a non-iterative version of rejection sampling instead of speculative sampling - specifically go through steps 1 and 2 above, and otherwise sample from an *unmodified* \( p(x) \) directly. That would be much less efficient than our method though. Specifically, the expected accept probability here is \( E_{x \sim q(x)} \frac{p(x)}{Mq(x)} = \sum_x p(x) \min_{x'} \frac{q(x')}{p(x')} \le \sum_x p(x) \min(1, \frac{q(x)}{p(x)}) = \sum_x \min(p(x), q(x)) = \alpha \) is (potentially much) lower than the expected accept probability in our method \( \alpha \).

### A.3. Theoretical Predictions vs. Empirical Runtimes

Table 4 compares the expected runtime improvements based on Theorem 3.8 to the empirically measured runtimes from Table 2. We estimated the values of \( c \) for the various models based on profiler traces. We can see that the theoretical predictions mostly match the measured runtimes. The larger differences are due to: (1) optimization differences between our implementation and the baseline, and (2) the simplifying assumption that the \( \beta \)s are i.i.d. being only an approximation (see Section 3.1).

<!-- page 11 -->


## Appendix A.4–A.5

### Fast Inference from Transformers via Speculative Decoding

Table 4. Expected improvement factor (EXP) vs. empirically measured improvement factor (EMP).

| TASK | \(M_q\) | TEMP | \(\gamma\) | \(\alpha\) | \(c\) | EXP | EMP |
| :--- | :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| ENDE | T5-SMALL | 0 | 7 | 0.75 | 0.02 | 3.2 | 3.4 |
| ENDE | T5-BASE | 0 | 7 | 0.8 | 0.04 | 3.3 | 2.8 |
| ENDE | T5-LARGE | 0 | 7 | 0.82 | 0.11 | 2.5 | 1.7 |
| ENDE | T5-SMALL | 1 | 7 | 0.62 | 0.02 | 2.3 | 2.6 |
| ENDE | T5-BASE | 1 | 5 | 0.68 | 0.04 | 2.4 | 2.4 |
| ENDE | T5-LARGE | 1 | 3 | 0.71 | 0.11 | 2.0 | 1.4 |
| CNNDM | T5-SMALL | 0 | 5 | 0.65 | 0.02 | 2.4 | 3.1 |
| CNNDM | T5-BASE | 0 | 5 | 0.73 | 0.04 | 2.6 | 3.0 |
| CNNDM | T5-LARGE | 0 | 3 | 0.74 | 0.11 | 2.0 | 2.2 |
| CNNDM | T5-SMALL | 1 | 5 | 0.53 | 0.02 | 1.9 | 2.3 |
| CNNDM | T5-BASE | 1 | 3 | 0.55 | 0.04 | 1.8 | 2.2 |
| CNNDM | T5-LARGE | 1 | 3 | 0.56 | 0.11 | 1.6 | 1.7 |

#### A.4. Application to Beam Search

Our method can be applied, with some performance penalty, to beam search sampling. Given the original beam width \(w\), we can perform beam search with the approximation model \(M_q\) and beam width \(u \ge w\) for \(\gamma\) steps. Then, we can use \(M_p\) to check all of the candidates in parallel (costing a compute budget of \((w + u\gamma)\) runs of \(M_p\)). Finally, for each step, we can accept the guesses of \(M_q\) as long as \(top_w(M_p) \subseteq top_u(M_q)\) to get identical results to regular beam search with \(M_p\) alone (with a more elaborate procedure we could also accept cases where the candidates we got happen to have higher probabilities than those of \(M_p\) alone). The analysis of our method in this setting is more involved and we leave it for future work.

#### A.5. Lenience

A strong property of Algorithm 1 is that the output distribution is guaranteed to remain unchanged. That said, if we're willing to allow some changes, with nice guarantees, we can get further inference speed improvements. To further motivate this, note that when we train two models with identical architectures and sizes on the same dataset, the generated probability distributions will not be identical, so some lenience might make sense. Note that the results in this paper except for this section use the strictest version of Algorithm 1 and don't allow lenience of any kind.

We could include a lenience parameter \(l \in [0, 1]\) and multiply \(q(x)\) by \(l\) before comparing with \(p(x)\) in Algorithm 1. This still maintains the nice guarantee that no token can be sampled with probability greater than \(\frac{p(x)}{l}\). This means for example, that with \(l = \frac{1}{10}\) no token can be sampled with more than 10X its ground truth probability, so we can guarantee that extremely rare tokens will remain extremely rare (there is no guarantee on the minimum probability, so lenience could hurt the diversity of the samples).

Specifically, with a lenience factor \(l\) we have \(\alpha = E_{x \sim q(x)} \begin{cases} 1 & lq(x) \le p(x) \\ \frac{p(x)}{lq(x)} & lq(x) > p(x) \end{cases} = E_{x \sim q(x)} \frac{p(x)}{max(p(x), lq(x))} = \sum_x \frac{p(x)q(x)}{max(p(x), lq(x))} = \frac{1}{l} \sum_x min(p(x), lq(x)) = \sum_x min(\frac{p(x)}{l}, q(x))\).

Table 5 shows \(\alpha\) values for different values of \(l\) when \(M_p\) is T5-XXL (11B) and \(M_q\) is T5-small (77M). With \(c = 0.015\), using lenience values of 1, 0.5, 0.3, and 0.1 (meaning that no token can be sampled with probability greater than 1X, 2X, 3X and 10X of the ground truth) we get improvement factors of 2.5X, 3.1X, 3.6X, and 5X respectively.

Table 5. \(\alpha\) values for various values of \(l\) with standard sampling where \(M_p\) is T5-XXL (11B) on the EnDe translation task.

| \(M_q\) | \(l=1\) | \(l=0.5\) | \(l=0.3\) | \(l=0.1\) |
| :--- | :---: | :---: | :---: | :---: |
| UNIGRAM | 0.07 | 0.1 | 0.11 | 0.16 |
| BIGRAM | 0.19 | 0.23 | 0.25 | 0.32 |
| T5-SMALL (77M) | 0.62 | 0.71 | 0.76 | 0.84 |
| T5-BASE (250M) | 0.68 | 0.8 | 0.83 | 0.90 |

<!-- page 12 -->

Note that when using temperature = 0 (i.e. argmax sampling), we can no longer use lenience as above. Instead, we could allow some lenience before standardizing the distributions. For example, we could accept the token \(x\) sampled from \(M_q\) in case \(p(x) \leq l \cdot max(p)\). In this case, we measure similar empirical increases in \(\alpha\) values to those with temperature = 1. For example, when using lenience values of 1, 0.5, 0.3, and 0.1 for \(M_p\) T5-XXL \(M_q\) T5-small for English-German translation, we get \(\alpha\) values of 0.75, 0.75, 0.8, 0.87. Taking for example \(c = 0.015\) and \(\gamma = 8\) we get speed improvement factors of 3.3X, 3.3X, 3.9X, and 4.9X respectively[^7].

[^7]: In this case, unlike in the standard sampling case shown in Table 5, a lenience factor of 0.5 doesn't improve the speed-up.

I'd like to read the above paper.  Currently, it is all in one message, can you help me break it into separate messages, with either just a single subsection per message, or just a few paragraphs for longer sections?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

I'd be happy to help with that. Let me first read the current message to see exactly what we're working with, then propose a split plan.



<details class='tool-usage-details' markdown='1'>
<summary><code>py(code=&quot;await view_msg(&#x27;3912b37c&#x27;, nums=True)&quot;)→&quot;     1 │ md_file_name = &#x27;leviathan-2023.…&quot;</code></summary>

```json
{
  "id": "chatcmpl-tool-99cca0f5a64eeda1",
  "server": false,
  "call": {
    "function": "py",
    "arguments": {
      "code": "await view_msg('3912b37c', nums=True)"
    }
  },
  "result": "     1 │ md_file_name = 'leviathan-2023.md'\n     2 │ Markdown(Path(md_file_name).read_text())"
}
```

</details>





<details class='tool-usage-details' markdown='1'>
<summary><code>py(code=&quot;await read_msgid(&#x27;3912b37c&#x27;)&quot;)→&quot;{&#x27;id&#x27;: &#x27;3912b37c&#x27;, &#x27;meta&#x27;: {}, &#x27;time_run…&quot;</code></summary>

```json
{
  "id": "chatcmpl-tool-a6ed46c91090c2f4",
  "server": false,
  "call": {
    "function": "py",
    "arguments": {
      "code": "await read_msgid('3912b37c')"
    }
  },
  "result": "<TRUNCATED>…12b37c', 'meta': {}, 'time_run': '2026-07-29T05:56:47.106828+00:00', 'bookmark': False, 'heading_collapsed': False, 'i_clamp': False, 'o_clamp': False, 'run': None, 'skipped': 0, 'pinned': 0, 'o_collapsed': 0, 'i_collapsed': 0, 'oob': None, 'scroll': False, 'order': 1, 'content': \"md_file_name = 'leviathan-2023.md'\\nMarkdown(Path(md_file_name).read_text())\", 'output': '<div class=\"prose\" markdown=\"1\">\\n\\n<!-- page 0 -->\\n\\n# Fast Inference from Transformers via Speculative Decoding\\n\\n**Yaniv Leviathan** \\\\(^{*1}\\\\) **Matan Kalman** \\\\(^{*1}\\\\) **Yossi Matias** \\\\(^{1}\\\\)\\n\\n### Abstract\\n\\nInference from large autoregressive models like Transformers is slow - decoding \\\\(K\\\\) tokens takes \\\\(K\\\\) serial runs of the model. In this work we introduce *speculative decoding* - an algorithm to sample from autoregressive models faster *without any changes to the outputs*, by computing several tokens in parallel. At the heart of our approach lie the observations that (1) hard language-modeling tasks often include easier subtasks that can be approximated well by more efficient models, and (2) using speculative execution and a novel sampling method, we can make exact decoding from the large models faster, by running them in parallel on the outputs of the approximation models, potentially generating several tokens concurrently, and without changing the distribution. Our method can accelerate existing off-the-shelf models without retraining or architecture changes. We demonstrate it on T5-XXL and show a **2X-3X** acceleration compared to the standard T5X implementation, with identical outputs.\\n\\n## 1. Introduction\\n\\nLarge autoregressive models, notably large Transformers (Vaswani et al., 2017), are much more capable than smaller models, as is evidenced countless times in recent years e.g., in the text or image domains, like GPT-3 (Brown et al., 2020), LaMDA (Thoppilan et al., 2022), Parti (Yu et al., 2022), and PaLM (Chowdhery et al., 2022). Unfortunately, a single decode step from these larger models is significantly slower than a step from their smaller counterparts, and making things worse, these steps are done serially - decoding \\\\(K\\\\) tokens takes \\\\(K\\\\) serial runs of the model.\\n\\nGiven the importance of large autoregressive models and specifically large Transformers, several approaches were developed to make inference from them faster. Some approaches aim to reduce the inference cost for *all* inputs equally (e.g. Hinton et al., 2015; Jaszczur et al., 2021; Hubara et al., 2016; So et al., 2021; Shazeer, 2019). Other approaches stem from the observation that not all inference steps are born alike - some require a very large model, while others can be approximated well by more efficient models. These *adaptive computation* methods (e.g. Han et al., 2021; Sukhbaatar et al., 2019; Schuster et al., 2021; Scardapane et al., 2020; Bapna et al., 2020; Elbayad et al., 2019; Schwartz et al., 2020) aim to use less compute resources for easier inference steps. While many of these solutions have proven extremely effective in practice, they usually require changing the model architecture, changing the training-procedure and re-training the models, and don\\'t maintain identical outputs.\\n\\nThe key observation above, that some inference steps are \"harder\" and some are \"easier\", is also a key motivator for our work. We additionally observe that inference from large models is often not bottlenecked on arithmetic operations, but rather on memory bandwidth and communication, so additional computation resources might be available. Therefore we suggest increasing concurrency as a complementary approach to using an adaptive amount of computation. Specifically, we are able to accelerate inference without changing the model architectures, without changing the training-procedures or needing to re-train the models, and without changing the model output distribution. This is accomplished via *speculative execution*.\\n\\nSpeculative execution (Burton, 1985; Hennessy & Patterson, 2012) is an optimization technique, common in processors, where a task is performed in parallel to verifying if it\\'s actually needed - the payoff being increased concurrency. A well-known example of speculative execution is branch prediction. For speculative execution to be effective, we need an efficient mechanism to suggest tasks to execute that are likely to be needed. In this work, we generalize speculative execution to the stochastic setting - where a task *might* be needed with some probability. Applying this to decoding from autoregressive models like Transformers, we sample generations from more efficient *approximation models* as speculative prefixes for the slower *target models*. With a novel sampling method, *speculative sampling*, we maximize the probability of these speculative tasks to\\n\\n\\\\rule{0.5\\\\textwidth}{0.4pt}  \\n\\\\(^{*}\\\\)Equal contribution \\\\(^{1}\\\\)Google Research, Mountain View, CA, USA. Correspondence to: Yaniv …</TRUNCATED>"
}
```

</details>





I can see the full paper content. It's a code message that reads `leviathan-2023.md` and renders it. I'd propose splitting it into separate note messages, each containing one subsection. Here's the breakdown I'd suggest:

1. Title, Authors, Abstract
2. §1 Introduction
3. §2.1 Overview
4. §2.2 Standardized Sampling
5. §2.3 Speculative Sampling (with Algorithm 1)
6. §3.1 Number of Generated Tokens
7. §3.2 Calculating α
8. §3.3 Walltime Improvement
9. §3.4 Number of Arithmetic Operations
10. §3.5 Choosing γ
11. §3.6 Approximation Models
12. §4.1 Empirical Walltime Improvement
13. §4.2 Empirical α Values
14. §5 Related Work
15. §6 Discussion
16. Acknowledgments + References
17. Appendix A.1–A.5 (split into 2–3 messages)

That's roughly 18–19 note messages. I'd add them all after the original code message (keeping it as-is so the source remains), and each note would be plain markdown so it renders nicely.

Should I go ahead with this plan? And would you prefer the references as one message or split further?

Go ahead with the plan

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me read the source file to get the exact content for splitting:



<details class='tool-usage-details' markdown='1'>
<summary><code>py(code=&quot;txt = Path(&#x27;leviathan-2023.md&#x27;).read_tex…&quot;)→&quot;51498 chars\n&lt;!-- page 0 --&gt;\n\n# Fast Infe…&quot;</code></summary>

```json
{
  "id": "chatcmpl-tool-8077a68f2641f44f",
  "server": false,
  "call": {
    "function": "py",
    "arguments": {
      "code": "txt = Path('leviathan-2023.md').read_text()\nprint(len(txt), 'chars')\nprint(txt[:200])"
    }
  },
  "result": "51498 chars\n<!-- page 0 -->\n\n# Fast Inference from Transformers via Speculative Decoding\n\n**Yaniv Leviathan** \\(^{*1}\\) **Matan Kalman** \\(^{*1}\\) **Yossi Matias** \\(^{1}\\)\n\n### Abstract\n\nInference from large aut"
}
```

</details>





Now let me split the content into sections. I'll identify section boundaries by the markdown headings:



<details class='tool-usage-details' markdown='1'>
<summary><code>py(code=&quot;lines = txt.split(&#x27;\\n&#x27;)\nsections = []\ncu…&quot;)→&quot; 0 | preamble                           …&quot;</code></summary>

```json
{
  "id": "chatcmpl-tool-801f75beaa50e8e4",
  "server": false,
  "call": {
    "function": "py",
    "arguments": {
      "code": "lines = txt.split('\\n')\nsections = []\ncurrent = []\ncurrent_title = \"preamble\"\nfor line in lines:\n    # Detect section headings\n    if line.startswith('# ') or line.startswith('## ') or line.startswith('### '):\n        if current:\n            sections.append((current_title, '\\n'.join(current)))\n        current_title = line.strip()\n        current = [line]\n    else:\n        current.append(line)\nif current:\n    sections.append((current_title, '\\n'.join(current)))\n\nfor i, (title, content) in enumerate(sections):\n    print(f\"{i:2d} | {title[:80]:80s} | {len(content):6d} chars\")"
    }
  },
  "result": " 0 | preamble                                                                         |     16 chars\n 1 | # Fast Inference from Transformers via Speculative Decoding                      |    144 chars\n 2 | ### Abstract                                                                     |   1022 chars\n 3 | ## 1. Introduction                                                               |   5878 chars\n 4 | ## 2. Speculative Decoding                                                       |     27 chars\n 5 | ### 2.1. Overview                                                                |   1278 chars\n 6 | ### 2.2. Standardized Sampling                                                   |    681 chars\n 7 | ## 2.3. Speculative Sampling                                                     |   2143 chars\n 8 | ## 3. Analysis                                                                   |     15 chars\n 9 | ### 3.1. Number of Generated Tokens                                              |   1149 chars\n10 | ### 3.2. Calculating \\( \\alpha \\)                                                |    730 chars\n11 | ### Fast Inference from Transformers via Speculative Decoding                    |    661 chars\n12 | ### 3.3. Walltime Improvement                                                    |   2519 chars\n13 | ### 3.4. Number of Arithmetic Operations                                         |   2160 chars\n14 | ### 3.5. Choosing \\( \\gamma \\)                                                   |    239 chars\n15 | ## Fast Inference from Transformers via Speculative Decoding                     |   1796 chars\n16 | ### 3.6. Approximation Models                                                    |   3638 chars\n17 | ### 4. Experiments                                                               |   4231 chars\n18 | ### 5. Related work                                                              |   4476 chars\n19 | ### 6. Discussion                                                                |   3486 chars\n20 | ### Acknowledgments                                                              |    521 chars\n21 | ### References                                                                   |   7236 chars\n22 | ## A. Appendix                                                                   |     15 chars\n23 | ### A.1. Correctness of Speculative Sampling                                     |   1042 chars\n24 | ### A.2. Speculative Sampling vs. Rejection Sampling                             |    895 chars\n25 | ### A.3. Theoretical Predictions vs. Empirical Runtimes                          |    586 chars\n26 | ### Fast Inference from Transformers via Speculative Decoding                    |   4888 chars"
}
```

</details>





Good — I can see the structure. I'll merge the tiny header-only sections with their first subsection, and fold the repeated page-header fragments into their preceding section. Let me build the final message list:



<details class='tool-usage-details' markdown='1'>
<summary><code>py(code=&quot;# Build merged sections\nmerged = []\n\n# 1…&quot;)→&quot; 0 | Title, Authors &amp; Abstract          …&quot;</code></summary>

```json
{
  "id": "chatcmpl-tool-9a70c1718fdfcdb7",
  "server": false,
  "call": {
    "function": "py",
    "arguments": {
      "code": "# Build merged sections\nmerged = []\n\n# 1. Title + Authors + Abstract (0,1,2)\nmerged.append(('Title, Authors & Abstract', \n    '\\n'.join(sections[i][1] for i in [0,1,2])))\n\n# 2. §1 Introduction (3)\nmerged.append(('§1 Introduction', sections[3][1]))\n\n# 3. §2.1 Overview (4+5)\nmerged.append(('§2 Speculative Decoding — §2.1 Overview', \n    '\\n'.join(sections[i][1] for i in [4,5])))\n\n# 4. §2.2 Standardized Sampling (6)\nmerged.append(('§2.2 Standardized Sampling', sections[6][1]))\n\n# 5. §2.3 Speculative Sampling (7)\nmerged.append(('§2.3 Speculative Sampling', sections[7][1]))\n\n# 6. §3.1 Number of Generated Tokens (8+9)\nmerged.append(('§3 Analysis — §3.1 Number of Generated Tokens',\n    '\\n'.join(sections[i][1] for i in [8,9])))\n\n# 7. §3.2 Calculating α (10+11 - 11 is page header continuation)\nmerged.append(('§3.2 Calculating α',\n    '\\n'.join(sections[i][1] for i in [10,11])))\n\n# 8. §3.3 Walltime Improvement (12)\nmerged.append(('§3.3 Walltime Improvement', sections[12][1]))\n\n# 9. §3.4 Number of Arithmetic Operations (13)\nmerged.append(('§3.4 Number of Arithmetic Operations', sections[13][1]))\n\n# 10. §3.5 Choosing γ (14+15)\nmerged.append(('§3.5 Choosing γ',\n    '\\n'.join(sections[i][1] for i in [14,15])))\n\n# 11. §3.6 Approximation Models (16)\nmerged.append(('§3.6 Approximation Models', sections[16][1]))\n\n# 12. §4 Experiments (17)\nmerged.append(('§4 Experiments', sections[17][1]))\n\n# 13. §5 Related Work (18)\nmerged.append(('§5 Related Work', sections[18][1]))\n\n# 14. §6 Discussion (19)\nmerged.append(('§6 Discussion', sections[19][1]))\n\n# 15. Acknowledgments + References (20+21)\nmerged.append(('Acknowledgments & References',\n    '\\n'.join(sections[i][1] for i in [20,21])))\n\n# 16. Appendix A.1-A.3 (22+23+24+25)\nmerged.append(('Appendix A.1–A.3',\n    '\\n'.join(sections[i][1] for i in [22,23,24,25])))\n\n# 17. Appendix A.4-A.5 (26)\nmerged.append(('Appendix A.4–A.5', sections[26][1]))\n\nfor i, (title, content) in enumerate(merged):\n    print(f\"{i:2d} | {title[:60]:60s} | {len(content):6d} chars\")"
    }
  },
  "result": " 0 | Title, Authors & Abstract                                    |   1184 chars\n 1 | §1 Introduction                                              |   5878 chars\n 2 | §2 Speculative Decoding — §2.1 Overview                      |   1306 chars\n 3 | §2.2 Standardized Sampling                                   |    681 chars\n 4 | §2.3 Speculative Sampling                                    |   2143 chars\n 5 | §3 Analysis — §3.1 Number of Generated Tokens                |   1165 chars\n 6 | §3.2 Calculating α                                           |   1392 chars\n 7 | §3.3 Walltime Improvement                                    |   2519 chars\n 8 | §3.4 Number of Arithmetic Operations                         |   2160 chars\n 9 | §3.5 Choosing γ                                              |   2036 chars\n10 | §3.6 Approximation Models                                    |   3638 chars\n11 | §4 Experiments                                               |   4231 chars\n12 | §5 Related Work                                              |   4476 chars\n13 | §6 Discussion                                                |   3486 chars\n14 | Acknowledgments & References                                 |   7758 chars\n15 | Appendix A.1–A.3                                             |   2541 chars\n16 | Appendix A.4–A.5                                             |   4888 chars"
}
```

</details>





17 messages, all reasonable sizes. Now I'll add them as note messages after the original code message, chaining each one from the previous:



<details class='tool-usage-details' markdown='1'>
<summary><code>py(code=&quot;prev_id = &#x27;3912b37c&#x27;\nids = []\nfor title,…&quot;)→&quot;[&#x27;63c406bd&#x27;, &#x27;f35c977f&#x27;, &#x27;f3c52cb8&#x27;, &#x27;28…&quot;</code></summary>

```json
{
  "id": "chatcmpl-tool-a2e144df0e684312",
  "server": false,
  "call": {
    "function": "py",
    "arguments": {
      "code": "prev_id = '3912b37c'\nids = []\nfor title, content in merged:\n    # Add a heading line for context\n    msg_content = f\"## {title}\\n\\n{content}\"\n    prev_id = await add_msg(msg_content, msg_type='note', placement='add_after', id=prev_id)\n    ids.append(prev_id)\nids"
    }
  },
  "result": "['63c406bd', 'f35c977f', 'f3c52cb8', '282a2dea', 'c2828929', '2841fe75', '21b51fe3', '3e827f39', '9c4c4f0b', 'd2c0acfa', '0db84f4d', '99d4c67e', 'd6dac405', 'eeecc669', 'd6541464', 'adc64675', 'e6e1bedd']"
}
```

</details>

All 17 note messages added. Here they are in order:

1. #_63c406bd — Title, Authors & Abstract
2. #_f35c977f — §1 Introduction
3. #_f3c52cb8 — §2 Speculative Decoding — §2.1 Overview
4. #_282a2dea — §2.2 Standardized Sampling
5. #_c2828929 — §2.3 Speculative Sampling
6. #_2841fe75 — §3 Analysis — §3.1 Number of Generated Tokens
7. #_21b51fe3 — §3.2 Calculating α
8. #_3e827f39 — §3.3 Walltime Improvement
9. #_9c4c4f0b — §3.4 Number of Arithmetic Operations
10. #_d2c0acfa — §3.5 Choosing γ
11. #_0db84f4d — §3.6 Approximation Models
12. #_99d4c67e — §4 Experiments
13. #_d6dac405 — §5 Related Work
14. #_eeecc669 — §6 Discussion
15. #_d6541464 — Acknowledgments & References
16. #_adc64675 — Appendix A.1–A.3
17. #_e6e1bedd — Appendix A.4–A.5

The original code message (#_3912b37c) is preserved as-is so the source file reference remains intact. Each note renders the markdown directly, so the math, tables, and figures should display properly.

A different animations of speculative decoding:
- https://www.adaptive-ml.com/post/speculative-decoding-visualized

Impact of Speculative Decoding:
- https://research.google/blog/looking-back-at-speculative-decoding/